# COMP8851 — PMP Benchmark

## Author-Reproduction Evidence + Unified Controlled Benchmark

**Model:** PMP — Partitioning Message Passing for Graph Fraud Detection  
**Official repository:** `Xtra-Computing/PMP`  
**Frozen source commit:** `3f7629f6c180891a0bc1bba3c66d94d288a1ddae`

### Existing author-reproduction evidence

PMP already passed the initial author-code feasibility/reproduction gate on YelpChi in an earlier Kaggle P100 session:

- supplied checkpoint evaluation completed;
- an independent scratch run completed;
- author-mode and scratch results were recorded separately;
- the exact repository commit and isolated environment were preserved.

That prior evidence is not overwritten here.

### Purpose of this notebook

This notebook moves PMP into the **controlled Kaggle T4 benchmark** while keeping author reproduction separate.

The controlled workflow is:

1. verify T4 GPU 0 and internet;
2. freeze the official PMP source;
3. audit source-code split, seed, threshold and label-partition behaviour before editing;
4. build an isolated author-compatible Python/CUDA environment;
5. attach and verify the canonical dataset and frozen project split;
6. run a one-epoch unified smoke;
7. implement only the minimum non-architectural adapter required by the shared protocol;
8. tune on TR40 only with validation AUPRC;
9. freeze the winner;
10. run TR40/TR30/TR20/TR10 with seeds 2, 42 and 72;
11. record raw epoch timing, inference latency and peak GPU memory;
12. package evidence immediately after each dataset.

### Dataset order

1. **YelpChi** — first controlled dataset; prior author reproduction already passed.
2. **Amazon** — second native repository dataset.
3. **T-Finance** — third native repository dataset.
4. **T-Social** — compatibility audit after the three native paths; README mentions the data but the frozen repository does not supply a `tsocial.yml` config.
5. **FDCompCN** — input-adapter feasibility only.
6. **Elliptic** — input-adapter feasibility only; chronology must be preserved.

No dataset-model combination will be forced by redesigning PMP.


## Step 1 — Kaggle Runtime Preflight

This step uses no dataset and does not import PyTorch into the notebook kernel.

It verifies:

- NVIDIA T4 availability;
- GPU 0 policy;
- internet/GitHub access;
- the working directory;
- whether Python 3.10 or Conda is available for the isolated PMP environment.

**Do not attach datasets yet.**


In [1]:
# ============================================================
# STEP 1 — KAGGLE T4 / INTERNET / WORKSPACE PREFLIGHT
# No torch import in the notebook kernel.
# ============================================================

from pathlib import Path
import os, sys, json, time, shutil, subprocess, platform

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

ROOT = Path("/kaggle/working/comp8851_pmp")
DIRS = {
    "source": ROOT / "source",
    "env": ROOT / "env",
    "evidence": ROOT / "evidence",
    "logs": ROOT / "logs",
    "patches": ROOT / "patches",
    "results_reproduction": ROOT / "results" / "author-reproduction",
    "results_unified": ROOT / "results" / "unified",
    "shared": ROOT / "shared",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, timeout=30, check=False):
    p = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        check=False,
    )
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(cmd)}\n{p.stdout}"
        )
    return p

print("===== STEP 1 — PMP KAGGLE PREFLIGHT =====", flush=True)
print("Kernel Python :", sys.version.replace("\n", " "), flush=True)
print("Platform      :", platform.platform(), flush=True)
print("Workspace     :", ROOT, flush=True)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"], flush=True)

# GPU audit without importing torch.
gpu = run_cmd(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ],
    timeout=20,
)
print("\n===== NVIDIA-SMI =====", flush=True)
print(gpu.stdout.strip(), flush=True)

gpu_lines = [x.strip() for x in gpu.stdout.splitlines() if x.strip()]
gpu0_is_t4 = bool(gpu_lines) and ("T4" in gpu_lines[0])

# Internet / GitHub audit.
git_remote = run_cmd(
    ["git", "ls-remote", "https://github.com/Xtra-Computing/PMP.git", "HEAD"],
    timeout=25,
)
internet_ok = git_remote.returncode == 0 and len(git_remote.stdout.strip()) > 0

print("\nGitHub access :", "PASS" if internet_ok else "FAIL", flush=True)
if internet_ok:
    print("Remote HEAD   :", git_remote.stdout.strip(), flush=True)

tools = {
    "git": shutil.which("git"),
    "python3.10": shutil.which("python3.10"),
    "conda": shutil.which("conda"),
    "mamba": shutil.which("mamba"),
    "micromamba": shutil.which("micromamba"),
}
print("\n===== ENVIRONMENT TOOL DISCOVERY =====", flush=True)
for k, v in tools.items():
    print(f"{k:<12}: {v}", flush=True)

record = {
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "kernel_python": sys.version,
    "platform": platform.platform(),
    "cuda_visible_devices": os.environ["CUDA_VISIBLE_DEVICES"],
    "nvidia_smi_query": gpu.stdout.strip(),
    "gpu0_is_t4": gpu0_is_t4,
    "github_access": internet_ok,
    "tools": tools,
}
with (DIRS["evidence"] / "step1_preflight.json").open("w") as f:
    json.dump(record, f, indent=2)

print("\n===== STEP 1 GATE =====", flush=True)
if gpu.returncode == 0 and gpu0_is_t4 and internet_ok:
    print("PASS — controlled PMP runtime preflight.", flush=True)
    print("GPU policy: T4 GPU 0", flush=True)
    print("Dataset attached yet: NOT REQUIRED", flush=True)
    print("Ready for repository freeze: YES", flush=True)
else:
    print("FAIL — do not continue.", flush=True)
    if not gpu0_is_t4:
        print("Reason: GPU 0 is not an NVIDIA T4.", flush=True)
    if not internet_ok:
        print("Reason: GitHub/internet access is unavailable.", flush=True)


===== STEP 1 — PMP KAGGLE PREFLIGHT =====
Kernel Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform      : Linux-6.12.90+-x86_64-with-glibc2.35
Workspace     : /kaggle/working/comp8851_pmp
CUDA_VISIBLE_DEVICES: 0

===== NVIDIA-SMI =====
0, Tesla T4, 15360, 580.159.04
1, Tesla T4, 15360, 580.159.04

GitHub access : PASS
Remote HEAD   : 3f7629f6c180891a0bc1bba3c66d94d288a1ddae	HEAD

===== ENVIRONMENT TOOL DISCOVERY =====
git         : /usr/bin/git
python3.10  : /usr/bin/python3.10
conda       : None
mamba       : /usr/local/bin/mamba
micromamba  : None

===== STEP 1 GATE =====
PASS — controlled PMP runtime preflight.
GPU policy: T4 GPU 0
Dataset attached yet: NOT REQUIRED
Ready for repository freeze: YES


## Step 2 — Freeze the Official PMP Repository

The official repository is cloned and checked out at the exact commit already preserved by the earlier PMP reproduction.

This step records:

- repository URL;
- exact commit;
- clean working-tree status;
- repository file tree;
- frozen README, requirements and native dataset configs.

No dataset is required.


In [2]:
# ============================================================
# STEP 2 — FREEZE OFFICIAL PMP REPOSITORY IDENTITY
# ============================================================

from pathlib import Path
import subprocess, shutil, json

PMP_REPO_URL = "https://github.com/Xtra-Computing/PMP.git"
PMP_COMMIT = "3f7629f6c180891a0bc1bba3c66d94d288a1ddae"
REPO = DIRS["source"] / "PMP"

print("===== STEP 2 — PMP SOURCE FREEZE =====", flush=True)

if REPO.exists():
    print("Removing previous working clone only:", REPO, flush=True)
    shutil.rmtree(REPO)

def git(args, cwd=None, timeout=120):
    p = subprocess.run(
        ["git", *args],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        check=False,
    )
    print(p.stdout, end="" if p.stdout.endswith("\n") else "\n", flush=True)
    if p.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed with code {p.returncode}")
    return p.stdout.strip()

git(["clone", "--no-checkout", PMP_REPO_URL, str(REPO)], timeout=180)
git(["checkout", "--detach", PMP_COMMIT], cwd=REPO)
head = git(["rev-parse", "HEAD"], cwd=REPO)
remote = git(["remote", "get-url", "origin"], cwd=REPO)
status = git(["status", "--porcelain"], cwd=REPO)

assert head == PMP_COMMIT, f"Commit mismatch: {head}"
assert remote.rstrip("/") in {
    "https://github.com/Xtra-Computing/PMP.git",
    "https://github.com/Xtra-Computing/PMP",
}, remote
assert status == "", f"Fresh clone is unexpectedly dirty:\n{status}"

source_evidence = DIRS["evidence"] / "source"
source_evidence.mkdir(parents=True, exist_ok=True)

(source_evidence / "repository_commit.txt").write_text(head + "\n")
(source_evidence / "repository_remote.txt").write_text(remote + "\n")
(source_evidence / "repository_tree.txt").write_text(
    git(["ls-tree", "-r", "--name-only", "HEAD"], cwd=REPO) + "\n"
)

for rel in ["requirements.txt", "README.md", "config/yelp.yml",
            "config/amazon.yml", "config/tfinance.yml"]:
    src = REPO / rel
    if src.exists():
        dst = source_evidence / rel.replace("/", "__")
        shutil.copy2(src, dst)

print("\n===== STEP 2 GATE =====", flush=True)
print("PASS — official PMP source frozen.", flush=True)
print("Repository:", remote, flush=True)
print("Commit    :", head, flush=True)
print("Working tree clean: YES", flush=True)
print("Author reproduction evidence will remain separate: YES", flush=True)


===== STEP 2 — PMP SOURCE FREEZE =====
Cloning into '/kaggle/working/comp8851_pmp/source/PMP'...
HEAD is now at 3f7629f upd readme
3f7629f6c180891a0bc1bba3c66d94d288a1ddae
https://github.com/Xtra-Computing/PMP.git

DataHelper/dataset.py
DataHelper/datasetHelper.py
DataHelper/sampler.py
README.md
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_21-39-04/amazon.yml
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_21-39-04/best_val_model_1234.pth
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_21-39-04/results.txt
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_21-45-07/amazon.yml
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_21-45-07/best_val_model_1234.pth
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_21-45-07/results.txt
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_23-32-59/amazon.yml
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_23-32-59/best_val_model_1234.pth
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_23-32-59/results.txt
checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_2

## Step 3 — PMP Source Audit Before Any Modification

PMP has model-specific behaviours that must be understood before unified benchmarking.

This gate checks:

- native config coverage;
- duplicate dependency pins;
- the stock CLI seed restriction;
- internal split generation;
- whether PMP's partition label tensor uses training labels only;
- checkpoint-selection behaviour;
- test-each-epoch behaviour;
- the stock decision-threshold rule.

A **PASS** here does not mean that stock PMP already matches the team protocol. It means the required controlled adaptations are explicitly identified before editing.


In [3]:
# ============================================================
# STEP 3 — PMP SOURCE / SPLIT / LEAKAGE / CLI AUDIT
# No training and no dataset loading.
# ============================================================

from pathlib import Path
import re, json, hashlib

print("===== STEP 3 — PMP SOURCE AUDIT =====", flush=True)

main_text = (REPO / "main.py").read_text(encoding="utf-8", errors="replace")
helper_text = (REPO / "DataHelper" / "datasetHelper.py").read_text(
    encoding="utf-8", errors="replace"
)
req_text = (REPO / "requirements.txt").read_text(
    encoding="utf-8", errors="replace"
)

config_dir = REPO / "config"
config_names = sorted(p.name for p in config_dir.glob("*.yml"))

def key_lines(text, patterns):
    lines = text.splitlines()
    out = []
    for i, line in enumerate(lines, start=1):
        if any(re.search(p, line, flags=re.I) for p in patterns):
            out.append(f"{i}: {line}")
    return out

print("\n===== CONFIG FILES PRESENT =====", flush=True)
for n in config_names:
    print(n, flush=True)

required_native = {"yelp.yml", "amazon.yml", "tfinance.yml"}
native_configs_ok = required_native.issubset(set(config_names))
tsocial_config_present = "tsocial.yml" in config_names

# Known requirements-file problem: two mutually redundant PyYAML pins.
pyyaml_lines = [
    x.strip() for x in req_text.splitlines()
    if x.strip().lower().startswith("pyyaml==")
]
duplicate_pyyaml_pin = len(pyyaml_lines) > 1

seed_match = re.search(
    r"add_argument\('--seed'.*?choices\s*=\s*\[([^\]]+)\]",
    main_text,
    flags=re.S,
)
seed_choices = []
if seed_match:
    seed_choices = [
        int(x.strip())
        for x in seed_match.group(1).split(",")
        if x.strip().lstrip("-").isdigit()
    ]
team_seeds_supported = all(s in seed_choices for s in [2, 42, 72])

train_only_partition = bool(re.search(
    r"label_unk\s*\[\s*self\.train_nid\s*\]\s*=\s*"
    r"self\.labels\s*\[\s*self\.train_nid\s*\]",
    helper_text,
))

internal_split_present = (
    "train_test_split" in helper_text
    and "random_state = 2" in helper_text
)
repo_pr_threshold_to_test = "thres = best_dev_results.best_pr_thres" in main_text
monitor_is_configurable = "getattr(dev_results, config['monitor'])" in main_text
test_each_epoch_guarded = "if config['test_each_epoch']" in main_text

print("\n===== REQUIREMENTS AUDIT =====", flush=True)
print("PyYAML pins:", pyyaml_lines, flush=True)
print("Blind `pip install -r requirements.txt` safe:",
      "NO" if duplicate_pyyaml_pin else "YES", flush=True)

print("\n===== CLI / SEED AUDIT =====", flush=True)
print("Repository --seed choices:", seed_choices, flush=True)
print("Team final seeds 2/42/72 accepted by stock CLI:",
      "YES" if team_seeds_supported else "NO", flush=True)

print("\n===== DATA / LABEL-PARTITION AUDIT =====", flush=True)
print("Native configs Yelp/Amazon/T-Finance:",
      "PASS" if native_configs_ok else "FAIL", flush=True)
print("T-Social config supplied by repo:",
      "YES" if tsocial_config_present else "NO", flush=True)
print("Internal random split logic present:",
      "YES" if internal_split_present else "NO", flush=True)
print("PMP label_unk populated from TRAIN IDs only:",
      "YES" if train_only_partition else "NO", flush=True)

print("\n===== SELECTION / THRESHOLD AUDIT =====", flush=True)
print("Validation monitor configurable:",
      "YES" if monitor_is_configurable else "NO", flush=True)
print("Test-each-epoch path guarded by config:",
      "YES" if test_each_epoch_guarded else "NO", flush=True)
print("Stock final test uses validation PR-curve threshold:",
      "YES" if repo_pr_threshold_to_test else "NO", flush=True)

print("\n===== IMPORTANT SOURCE LINES =====", flush=True)
for line in key_lines(
    main_text,
    [
        r"add_argument\('--seed'",
        r"add_argument\('--train_size'",
        r"add_argument\('--val_size'",
        r"config\['monitor'\]",
        r"best_pr_thres",
        r"test_each_epoch",
    ],
)[:30]:
    print("main.py", line, flush=True)

for line in key_lines(
    helper_text,
    [
        r"FraudDataset",
        r"train_test_split",
        r"random_state\s*=\s*2",
        r"label_unk",
        r"train_nid",
        r"tsocial",
    ],
)[:40]:
    print("datasetHelper.py", line, flush=True)

audit = {
    "repository_commit": PMP_COMMIT,
    "config_files": config_names,
    "native_configs_ok": native_configs_ok,
    "tsocial_config_present": tsocial_config_present,
    "requirements_pyyaml_pins": pyyaml_lines,
    "blind_requirements_install_safe": not duplicate_pyyaml_pin,
    "stock_seed_choices": seed_choices,
    "team_seeds_supported_stock_cli": team_seeds_supported,
    "internal_split_present": internal_split_present,
    "partition_labels_train_only": train_only_partition,
    "validation_monitor_configurable": monitor_is_configurable,
    "test_each_epoch_guarded": test_each_epoch_guarded,
    "stock_test_threshold_is_validation_pr_threshold": repo_pr_threshold_to_test,
    "unified_changes_required": [
        "reuse frozen project split IDs instead of stock/generated split IDs",
        "allow controlled training seeds 2, 42, 72",
        "select checkpoint by validation AUPRC",
        "apply common validation Macro-F1 threshold rule to exported scores",
        "record raw epoch timing, inference latency, and peak GPU memory",
    ],
}
(DIRS["evidence"] / "pmp_source_audit.json").write_text(
    json.dumps(audit, indent=2)
)

assert native_configs_ok
assert train_only_partition
assert monitor_is_configurable
assert test_each_epoch_guarded

print("\n===== STEP 3 GATE =====", flush=True)
print("PASS — PMP source audit completed.", flush=True)
print("Author-native configurations: YelpChi / Amazon / T-Finance", flush=True)
print("T-Social documented CLI/config status: CONDITIONAL (no tsocial.yml)", flush=True)
print("Validation/test label leakage in PMP partition labels: NOT FOUND", flush=True)
print("Unified split adapter required: YES", flush=True)
print("Unified seed CLI patch required: YES", flush=True)
print("Common threshold evaluator required: YES", flush=True)
print("Architecture redesign required: NO", flush=True)


===== STEP 3 — PMP SOURCE AUDIT =====

===== CONFIG FILES PRESENT =====
amazon.yml
tfinance.yml
yelp.yml

===== REQUIREMENTS AUDIT =====
PyYAML pins: ['PyYAML==6.0', 'PyYAML==6.0.1']
Blind `pip install -r requirements.txt` safe: NO

===== CLI / SEED AUDIT =====
Repository --seed choices: [0, 1, 1234]
Team final seeds 2/42/72 accepted by stock CLI: NO

===== DATA / LABEL-PARTITION AUDIT =====
Native configs Yelp/Amazon/T-Finance: PASS
T-Social config supplied by repo: NO
Internal random split logic present: YES
PMP label_unk populated from TRAIN IDs only: YES

===== SELECTION / THRESHOLD AUDIT =====
Validation monitor configurable: YES
Test-each-epoch path guarded by config: YES
Stock final test uses validation PR-curve threshold: YES

===== IMPORTANT SOURCE LINES =====
main.py 26:                'best_pr_thres',
main.py 39:     if config['test_each_epoch']:
main.py 60:             if config['test_each_epoch']:
main.py 65:                                                                 

## Step 4A — Build the Isolated PMP Environment

The Kaggle image exposes `/usr/bin/python3.10`, but that interpreter does not
include `ensurepip`, so the standard `python3.10 -m venv` method cannot create
a pip-enabled environment.

This is a Kaggle runtime packaging issue, not a PMP issue.

The controlled environment is therefore created with `virtualenv`, using the
same system Python 3.10 interpreter while keeping PMP isolated from the
Kaggle Python 3.12 kernel.

The environment uses the previously validated PMP software family:

- Python 3.10
- PyTorch 2.0.1 + CUDA 11.8
- DGL 1.1.1 + CUDA 11.8
- PyTorch Geometric 2.3.1
- torch-scatter 2.1.1
- torch-sparse 0.6.17
- NumPy 1.23.5
- SciPy 1.10.0
- scikit-learn 1.2.1
- NetworkX 2.8.4

The cell also checks for `libcusparse.so.11` before Step 4B. If the CUDA
runtime library is not already supplied by the installed wheels, the matching
CUDA 11 runtime packages are installed explicitly.

The Kaggle accelerator may expose two T4 GPUs, but PMP is restricted to
`CUDA_VISIBLE_DEVICES=0`, so only one T4 enters the controlled benchmark.

No dataset is required yet.

In [5]:
# ============================================================
# STEP 4A — BUILD ISOLATED PMP PYTHON 3.10 / CUDA 11.8 ENV
#
# FIX:
# Kaggle's /usr/bin/python3.10 exists but has no ensurepip.
# We therefore use virtualenv to seed pip into the Python 3.10
# environment instead of `python3.10 -m venv`.
#
# No dataset is required.
# ============================================================

from pathlib import Path

import json
import os
import shutil
import subprocess
import sys
import time


print(
    "===== STEP 4A — PMP ISOLATED ENVIRONMENT =====",
    flush=True
)


# ============================================================
# 0. REQUIRE PREVIOUS PREFLIGHT CELLS
# ============================================================

assert "ROOT" in globals(), (
    "ROOT is missing. Rerun Step 1 first."
)

assert "DIRS" in globals(), (
    "DIRS is missing. Rerun Step 1 first."
)


ENV_DIR = (
    DIRS["env"]
    / "pmp_py310"
)

ENV_PY = (
    ENV_DIR
    / "bin"
    / "python"
)

INSTALL_LOG = (
    DIRS["logs"]
    / "pmp_environment_install.log"
)

ENV_RECORD = (
    DIRS["evidence"]
    / "step4a_environment_build.json"
)


# Start a clean log for this corrected attempt.
INSTALL_LOG.parent.mkdir(
    parents=True,
    exist_ok=True
)

INSTALL_LOG.write_text(
    "PMP Step 4A environment installation log\n"
    "Corrected virtualenv-based Python 3.10 creation\n\n",
    encoding="utf-8"
)


# ============================================================
# 1. LOGGED COMMAND HELPER
# ============================================================

def run_logged(
    cmd,
    timeout=1800,
    env=None
):

    cmd = [
        str(x)
        for x in cmd
    ]

    print(
        "\n$",
        " ".join(cmd),
        flush=True
    )

    p = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        check=False,
        env=env
    )


    with INSTALL_LOG.open(
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            "$ "
            + " ".join(cmd)
            + "\n"
        )

        f.write(
            p.stdout
            + "\n"
        )


    tail = "\n".join(
        p.stdout.splitlines()[
            -30:
        ]
    )


    if tail:

        print(
            tail,
            flush=True
        )


    if p.returncode != 0:

        raise RuntimeError(
            f"Install command failed "
            f"({p.returncode}).\n"
            f"See: {INSTALL_LOG}\n\n"
            f"{tail}"
        )


    return p


# ============================================================
# 2. CLEAN THE PARTIALLY CREATED BROKEN ENV
# ============================================================

if ENV_DIR.exists():

    print(
        "Removing incomplete previous environment:",
        ENV_DIR,
        flush=True
    )

    shutil.rmtree(
        ENV_DIR
    )


# ============================================================
# 3. VERIFY SYSTEM PYTHON 3.10
# ============================================================

PY310 = shutil.which(
    "python3.10"
)


assert PY310 is not None, (
    "python3.10 is no longer available. "
    "STOP and send the Step 1 environment-tool output."
)


print(
    "System Python 3.10:",
    PY310,
    flush=True
)


py310_version = subprocess.check_output(
    [
        PY310,
        "--version"
    ],
    text=True,
    stderr=subprocess.STDOUT
).strip()


print(
    "Python version:",
    py310_version,
    flush=True
)


assert (
    "Python 3.10"
    in py310_version
)


# ============================================================
# 4. INSTALL VIRTUALENV BOOTSTRAP TOOL
#
# Only the virtualenv creator goes into the Kaggle kernel.
# PMP model dependencies stay inside ENV_DIR.
# ============================================================

print(
    "\n===== VIRTUALENV BOOTSTRAP =====",
    flush=True
)


run_logged(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "virtualenv==20.26.6"
    ],
    timeout=600
)


# ============================================================
# 5. CREATE PYTHON 3.10 ENV WITHOUT ENSUREPIP
# ============================================================

run_logged(
    [
        sys.executable,
        "-m",
        "virtualenv",
        "--python",
        PY310,
        str(ENV_DIR)
    ],
    timeout=300
)


assert ENV_PY.exists(), (
    f"Environment Python missing after virtualenv creation: "
    f"{ENV_PY}"
)


env_python_version = subprocess.check_output(
    [
        str(ENV_PY),
        "--version"
    ],
    text=True,
    stderr=subprocess.STDOUT
).strip()


print(
    "\nEnvironment Python:",
    ENV_PY,
    flush=True
)

print(
    "Environment version:",
    env_python_version,
    flush=True
)


assert (
    "Python 3.10"
    in env_python_version
)


# ============================================================
# 6. PIN BASIC PACKAGING TOOLS
# ============================================================

run_logged(
    [
        ENV_PY,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "pip==24.0",
        "setuptools==68.2.2",
        "wheel==0.41.3"
    ],
    timeout=600
)


# ============================================================
# 7. INSTALL PYTORCH 2.0.1 CUDA 11.8
#
# Do not specify +cu118 in the package name here.
# The CUDA 11.8 index supplies the correct wheel.
# ============================================================

run_logged(
    [
        ENV_PY,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "torch==2.0.1",
        "--index-url",
        "https://download.pytorch.org/whl/cu118"
    ],
    timeout=2400
)


# ============================================================
# 8. INSTALL DGL CUDA 11.8
# ============================================================

run_logged(
    [
        ENV_PY,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "dgl==1.1.1+cu118",
        "-f",
        "https://data.dgl.ai/wheels/cu118/repo.html"
    ],
    timeout=1800
)


# ============================================================
# 9. INSTALL PYG COMPILED EXTENSIONS
# ============================================================

run_logged(
    [
        ENV_PY,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "torch-scatter==2.1.1",
        "torch-sparse==0.6.17",
        "-f",
        "https://data.pyg.org/whl/"
        "torch-2.0.1+cu118.html"
    ],
    timeout=2400
)


# ============================================================
# 10. REMAINING AUTHOR-COMPATIBLE STACK
#
# We still do NOT blindly install requirements.txt because
# the frozen PMP repository contains conflicting PyYAML pins.
# ============================================================

run_logged(
    [
        ENV_PY,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",

        "torch-geometric==2.3.1",

        "numpy==1.23.5",
        "scipy==1.10.0",
        "scikit-learn==1.2.1",
        "imbalanced-learn==0.10.1",

        "networkx==2.8.4",

        "matplotlib==3.7.0",

        "ogb==1.3.6",

        "PyYAML==6.0.1",

        "torchmetrics==0.11.4",

        "tqdm==4.64.1",

        "tensorboard==2.13.0"
    ],
    timeout=2400
)


# ============================================================
# 11. LOCATE THE ENV SITE-PACKAGES
# ============================================================

site_output = subprocess.check_output(
    [
        str(ENV_PY),
        "-c",
        (
            "import site; "
            "print(site.getsitepackages()[0])"
        )
    ],
    text=True
).strip()


SITE_PATH = Path(
    site_output
)


assert SITE_PATH.exists()


print(
    "\nSite-packages:",
    SITE_PATH,
    flush=True
)


# ============================================================
# 12. CUDA LIBRARY DISCOVERY
# ============================================================

def collect_cuda_library_dirs():

    candidates = []


    # NVIDIA pip-package layout.
    nvidia_root = (
        SITE_PATH
        / "nvidia"
    )

    if nvidia_root.exists():

        candidates.extend(
            p
            for p in nvidia_root.glob(
                "*/lib"
            )
            if p.is_dir()
        )


    # PyTorch wheel layout.
    torch_lib = (
        SITE_PATH
        / "torch"
        / "lib"
    )

    if torch_lib.is_dir():

        candidates.append(
            torch_lib
        )


    # Kaggle/system CUDA fallbacks.
    for p in [

        Path(
            "/usr/local/cuda/lib64"
        ),

        Path(
            "/usr/local/cuda-11.8/lib64"
        ),

        Path(
            "/usr/lib/x86_64-linux-gnu"
        )

    ]:

        if p.is_dir():

            candidates.append(
                p
            )


    # Unique, order-preserving.
    unique = []

    seen = set()


    for p in candidates:

        text = str(
            p.resolve()
        )

        if text not in seen:

            seen.add(
                text
            )

            unique.append(
                p
            )


    return unique


def find_library(
    lib_dirs,
    pattern
):

    found = []


    for directory in lib_dirs:

        try:

            found.extend(
                directory.glob(
                    pattern
                )
            )

        except Exception:

            pass


    return sorted(
        {
            str(p.resolve())
            for p in found
            if p.exists()
        }
    )


CUDA_LIB_DIRS = (
    collect_cuda_library_dirs()
)


CUSPARSE = find_library(
    CUDA_LIB_DIRS,
    "libcusparse.so.11*"
)


print(
    "\n===== CUDA LIBRARY DISCOVERY =====",
    flush=True
)

print(
    "CUDA library directories:",
    len(CUDA_LIB_DIRS),
    flush=True
)


for p in CUDA_LIB_DIRS:

    print(
        " ",
        p,
        flush=True
    )


print(
    "libcusparse.so.11 candidates:",
    len(CUSPARSE),
    flush=True
)


for p in CUSPARSE[
    :10
]:

    print(
        " ",
        p,
        flush=True
    )


# ============================================================
# 13. FALLBACK — EXPLICIT CUDA 11 RUNTIME WHEELS
#
# Earlier PMP reproduction encountered DGL failing to locate
# libcusparse.so.11. Only install these packages if that
# library is currently absent.
# ============================================================

cuda_runtime_fallback_used = False


if not CUSPARSE:

    print(
        "\nlibcusparse.so.11 not found.",
        flush=True
    )

    print(
        "Installing matching CUDA 11 runtime wheels...",
        flush=True
    )


    run_logged(
        [
            ENV_PY,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",

            "nvidia-cuda-runtime-cu11==11.8.89",
            "nvidia-cublas-cu11==11.11.3.6",
            "nvidia-cusparse-cu11==11.7.5.86"
        ],
        timeout=2400
    )


    cuda_runtime_fallback_used = True


    CUDA_LIB_DIRS = (
        collect_cuda_library_dirs()
    )


    CUSPARSE = find_library(
        CUDA_LIB_DIRS,
        "libcusparse.so.11*"
    )


assert CUSPARSE, (
    "libcusparse.so.11 is still missing after CUDA runtime "
    "setup. STOP here and send the Step 4A output."
)


print(
    "\nFinal libcusparse candidates:",
    len(CUSPARSE),
    flush=True
)


for p in CUSPARSE[
    :10
]:

    print(
        " ",
        p,
        flush=True
    )


# ============================================================
# 14. CREATE PMP LAUNCHER
#
# Important for T4 x2:
# CUDA_VISIBLE_DEVICES=0 makes only ONE T4 visible to PMP.
# ============================================================

LAUNCHER = (
    ROOT
    / "pmp_python.sh"
)


LD_PREFIX = ":".join(
    str(p)
    for p in CUDA_LIB_DIRS
)


LAUNCHER.write_text(

    "#!/usr/bin/env bash\n"

    "set -euo pipefail\n"

    "export CUDA_VISIBLE_DEVICES=0\n"

    "export DGLBACKEND=pytorch\n"

    "export PYTHONUNBUFFERED=1\n"

    "export PYTHONNOUSERSITE=1\n"

    f'export LD_LIBRARY_PATH="'
    f'{LD_PREFIX}:'
    '${LD_LIBRARY_PATH:-}"\n'

    f'exec "{ENV_PY}" "$@"\n',

    encoding="utf-8"
)


LAUNCHER.chmod(
    0o755
)


assert LAUNCHER.exists()


# ============================================================
# 15. BASIC PYTHON/PIP SANITY — NO MODEL TRAINING
# ============================================================

sanity = subprocess.run(
    [
        str(LAUNCHER),
        "-c",
        (
            "import sys; "
            "print(sys.version); "
            "print(sys.executable)"
        )
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=60,
    check=False
)


print(
    "\n===== PYTHON SANITY =====",
    flush=True
)

print(
    sanity.stdout,
    flush=True
)


assert sanity.returncode == 0


# ============================================================
# 16. SAVE BUILD RECORD
# ============================================================

record = {

    "timestamp_utc":
        time.strftime(
            "%Y-%m-%dT%H:%M:%SZ",
            time.gmtime()
        ),

    "environment_path":
        str(
            ENV_DIR
        ),

    "environment_python":
        str(
            ENV_PY
        ),

    "python_version":
        env_python_version,

    "creation_method":
        (
            "virtualenv from Kaggle kernel "
            "targeting /usr/bin/python3.10"
        ),

    "reason_virtualenv_used":
        (
            "Kaggle /usr/bin/python3.10 lacks ensurepip; "
            "python3.10 -m venv cannot seed pip."
        ),

    "launcher":
        str(
            LAUNCHER
        ),

    "cuda_visible_devices":
        "0",

    "cuda_runtime_fallback_used":
        cuda_runtime_fallback_used,

    "cuda_library_dirs":
        [
            str(p)
            for p in CUDA_LIB_DIRS
        ],

    "libcusparse_candidates":
        CUSPARSE,

    "install_log":
        str(
            INSTALL_LOG
        )
}


ENV_RECORD.write_text(
    json.dumps(
        record,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n===== STEP 4A GATE =====",
    flush=True
)

print(
    "PASS — isolated PMP Python 3.10 environment built.",
    flush=True
)

print(
    "Environment creation: virtualenv / Python 3.10",
    flush=True
)

print(
    "ensurepip dependency: NOT REQUIRED",
    flush=True
)

print(
    "libcusparse.so.11 available: YES",
    flush=True
)

print(
    "CUDA runtime fallback used:",
    (
        "YES"
        if cuda_runtime_fallback_used
        else "NO"
    ),
    flush=True
)

print(
    "PMP launcher:",
    LAUNCHER,
    flush=True
)

print(
    "Controlled GPU exposure: CUDA_VISIBLE_DEVICES=0",
    flush=True
)

print(
    "Dataset required: NO",
    flush=True
)

print(
    "Ready for Step 4B CUDA/DGL gate: YES",
    flush=True
)

===== STEP 4A — PMP ISOLATED ENVIRONMENT =====
Removing incomplete previous environment: /kaggle/working/comp8851_pmp/env/pmp_py310
System Python 3.10: /usr/bin/python3.10
Python version: Python 3.10.12

===== VIRTUALENV BOOTSTRAP =====

$ /usr/bin/python3 -m pip install --no-cache-dir virtualenv==20.26.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 319.3 MB/s eta 0:00:00

$ /usr/bin/python3 -m virtualenv --python /usr/bin/python3.10 /kaggle/working/comp8851_pmp/env/pmp_py310
created virtual environment CPython3.10.12.final.0-64 in 879ms
  creator CPython3Posix(dest=/kaggle/working/comp8851_pmp/env/pmp_py310, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, setuptools=bundle, wheel=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==24.2, setuptools==75.1.0, wheel==0.44.0
  activators BashActivator,CShellActiva

## Step 4B — Environment and CUDA Gate

This gate verifies the exact installed versions and runs a real DGL CUDA graph operation.

Only after this cell ends with:

`PASS — isolated PMP environment is T4/CUDA ready.`

should the first dataset be attached.


In [6]:
# ============================================================
# STEP 4B — PMP ENVIRONMENT / CUDA / DGL GATE
# ============================================================

from pathlib import Path
import subprocess, json

print("===== STEP 4B — PMP ENVIRONMENT GATE =====", flush=True)

launcher = ROOT / "pmp_python.sh"
assert launcher.exists(), "PMP launcher missing; Step 4A did not complete."

probe = r"""
import sys, json
import torch
import dgl
import torch_geometric
import torch_scatter
import torch_sparse
import numpy
import scipy
import sklearn
import networkx
import yaml
import torchmetrics

record = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "dgl": dgl.__version__,
    "torch_geometric": torch_geometric.__version__,
    "torch_scatter": torch_scatter.__version__,
    "torch_sparse": torch_sparse.__version__,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "networkx": networkx.__version__,
    "pyyaml": yaml.__version__,
    "torchmetrics": torchmetrics.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count_visible": torch.cuda.device_count(),
    "gpu0": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu0_total_memory_bytes": (
        torch.cuda.get_device_properties(0).total_memory
        if torch.cuda.is_available() else None
    ),
}
print(json.dumps(record, indent=2))

assert record["python"].startswith("3.10.")
assert record["torch"].startswith("2.0.1")
assert str(record["dgl"]).startswith("1.1.1")
assert record["torch_geometric"] == "2.3.1"
assert record["numpy"] == "1.23.5"
assert record["scipy"] == "1.10.0"
assert record["sklearn"] == "1.2.1"
assert record["networkx"] == "2.8.4"
assert record["cuda_available"] is True
assert record["cuda_device_count_visible"] == 1
assert "T4" in record["gpu0"]

# Real DGL CUDA operation, not just import.
g = dgl.graph(([0, 1, 2], [1, 2, 0]))
g = g.to("cuda:0")
x = torch.arange(9, dtype=torch.float32, device="cuda:0").reshape(3, 3)
g.ndata["x"] = x
assert g.ndata["x"].is_cuda
print("DGL CUDA graph smoke: PASS")
"""

p = subprocess.run(
    [str(launcher), "-c", probe],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=180,
    check=False,
)
print(p.stdout, flush=True)

if p.returncode != 0:
    raise RuntimeError(
        "PMP environment gate failed. STOP here and send this output."
    )

# Save exact environment evidence.
env_dir = DIRS["evidence"] / "environment"
env_dir.mkdir(parents=True, exist_ok=True)

freeze = subprocess.run(
    [str(launcher), "-m", "pip", "freeze"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False
)
(env_dir / "pip_freeze.txt").write_text(freeze.stdout)

smi = subprocess.run(
    ["nvidia-smi"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False
)
(env_dir / "nvidia_smi.txt").write_text(smi.stdout)

(env_dir / "environment_probe.txt").write_text(p.stdout)

print("===== STEP 4B GATE =====", flush=True)
print("PASS — isolated PMP environment is T4/CUDA ready.", flush=True)
print("Controlled GPU visible to PMP: T4 GPU 0 only", flush=True)
print("Dataset attached yet: NO — attach only after this gate.", flush=True)
print("Ready for canonical YelpChi + frozen split input: YES", flush=True)


===== STEP 4B — PMP ENVIRONMENT GATE =====
{
  "python": "3.10.12",
  "torch": "2.0.1+cu118",
  "torch_cuda_build": "11.8",
  "dgl": "1.1.1+cu118",
  "torch_geometric": "2.3.1",
  "torch_scatter": "2.1.1+pt20cu118",
  "torch_sparse": "0.6.17+pt20cu118",
  "numpy": "1.23.5",
  "scipy": "1.10.0",
  "sklearn": "1.2.1",
  "networkx": "2.8.4",
  "pyyaml": "6.0.1",
  "torchmetrics": "0.11.4",
  "cuda_available": true,
  "cuda_device_count_visible": 1,
  "gpu0": "Tesla T4",
  "gpu0_total_memory_bytes": 15636037632
}
DGL CUDA graph smoke: PASS

===== STEP 4B GATE =====
PASS — isolated PMP environment is T4/CUDA ready.
Controlled GPU visible to PMP: T4 GPU 0 only
Dataset attached yet: NO — attach only after this gate.
Ready for canonical YelpChi + frozen split input: YES


## INPUT CHECKPOINT — YelpChi

Step 4B passed the isolated PMP CUDA/T4 environment gate.

The first controlled PMP dataset is now attached through Kaggle **Add Input**.

### Inputs required at this stage

1. Canonical `YelpChi.mat`
2. Frozen project split asset containing `yelp_seed2_nested_splits.npz`

### Inputs deliberately not attached yet

- Amazon
- T-Finance
- T-Social
- FDCompCN
- Elliptic

Each later dataset is attached only after the current dataset has completed
its benchmark and evidence package.

No training starts until the canonical dataset bytes and frozen split IDs pass
the next gates.

## Step 4C — Kaggle Input Mount Discovery

This step confirms what Kaggle actually mounted.

No hard-coded Kaggle dataset directory is assumed.

The cell looks recursively under `/kaggle/input` for:

- YelpChi `.mat` files;
- frozen split `.npz` files;
- ZIP files that may contain the frozen split;
- manifest files associated with the split package.

This is discovery only.

Dataset identity is verified by SHA256 in Step 5.

In [7]:
# ============================================================
# STEP 4C — KAGGLE INPUT MOUNT DISCOVERY
# No training.
# ============================================================

from pathlib import Path


INPUT_ROOT = Path(
    "/kaggle/input"
)


print(
    "===== STEP 4C — KAGGLE INPUT DISCOVERY =====",
    flush=True
)


assert INPUT_ROOT.exists(), (
    "/kaggle/input does not exist."
)


all_files = [

    p
    for p in INPUT_ROOT.rglob("*")
    if p.is_file()

]


print(
    "Total mounted input files:",
    len(all_files),
    flush=True
)


# ------------------------------------------------------------
# Only print potentially relevant files.
# ------------------------------------------------------------

interesting = []


for p in all_files:

    name = p.name.lower()

    if (
        p.suffix.lower()
        in {
            ".mat",
            ".npz",
            ".zip",
            ".json",
            ".txt"
        }
        and
        (
            "yelp" in name
            or
            "pmp" in name
            or
            "split" in name
            or
            "manifest" in name
        )
    ):

        interesting.append(
            p
        )


print(
    "\n===== RELEVANT MOUNTED FILES =====",
    flush=True
)


for p in sorted(
    interesting
):

    print(
        p,
        "|",
        f"{p.stat().st_size / (1024**2):.3f} MB",
        flush=True
    )


mat_candidates = [

    p
    for p in all_files
    if p.suffix.lower() == ".mat"

]


npz_candidates = [

    p
    for p in all_files
    if p.suffix.lower() == ".npz"

]


zip_candidates = [

    p
    for p in all_files
    if p.suffix.lower() == ".zip"

]


print(
    "\nMAT candidates:",
    len(mat_candidates),
    flush=True
)

print(
    "NPZ candidates:",
    len(npz_candidates),
    flush=True
)

print(
    "ZIP candidates:",
    len(zip_candidates),
    flush=True
)


print(
    "\n===== STEP 4C GATE =====",
    flush=True
)


if (
    len(
        mat_candidates
    ) >= 1
    and
    (
        len(
            npz_candidates
        ) >= 1
        or
        len(
            zip_candidates
        ) >= 1
    )
):

    print(
        "PASS — required input types are mounted.",
        flush=True
    )

    print(
        "Canonical identity verification performed: NO",
        flush=True
    )

    print(
        "PMP training started: NO",
        flush=True
    )

    print(
        "Ready for Step 5 SHA256 dataset/split gate: YES",
        flush=True
    )

else:

    print(
        "FAIL — required input types are not all mounted.",
        flush=True
    )

    if len(
        mat_candidates
    ) == 0:

        print(
            "Missing: YelpChi .mat input",
            flush=True
        )

    if (
        len(
            npz_candidates
        ) == 0
        and
        len(
            zip_candidates
        ) == 0
    ):

        print(
            "Missing: frozen split NPZ/ZIP input",
            flush=True
        )

===== STEP 4C — KAGGLE INPUT DISCOVERY =====
Total mounted input files: 6

===== RELEVANT MOUNTED FILES =====
/kaggle/input/datasets/pathikahmed0007/pmp-frozen-splits/MANIFEST.json | 0.005 MB
/kaggle/input/datasets/pathikahmed0007/pmp-frozen-splits/splits/amazon_seed2_nested_splits.npz | 0.023 MB
/kaggle/input/datasets/pathikahmed0007/pmp-frozen-splits/splits/tfinance_seed2_nested_splits.npz | 0.094 MB
/kaggle/input/datasets/pathikahmed0007/pmp-frozen-splits/splits/yelp_seed2_nested_splits.npz | 0.110 MB
/kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat | 198.056 MB

MAT candidates: 1
NPZ candidates: 3
ZIP candidates: 0

===== STEP 4C GATE =====
PASS — required input types are mounted.
Canonical identity verification performed: NO
PMP training started: NO
Ready for Step 5 SHA256 dataset/split gate: YES


## Step 5 — Canonical YelpChi and Frozen Split Gate

The mounted inputs are now verified by file content rather than by Kaggle
dataset/folder name.

### Canonical dataset identity

Expected YelpChi SHA256:

`fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42`

### Frozen project split identity

Expected YelpChi split SHA256:

`0ea0af36dfc5a3a1f381ea2e3168377e45826498b6063190185c673e9ec8c22b`

The frozen split must contain:

- split seed 2;
- TR40 = 18,381 nodes;
- TR30 = 13,785 nodes;
- TR20 = 9,190 nodes;
- TR10 = 4,595 nodes;
- validation = 9,191 nodes;
- test = 18,382 nodes;
- `TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40`;
- fixed, mutually disjoint validation and test IDs.

The canonical YelpChi graph is also checked for:

- 45,954 nodes;
- 32 input features;
- 6,677 fraud labels;
- 39,277 normal labels;
- three source relations.

No PMP training occurs in this step.

In [8]:
# ============================================================
# STEP 5 — CANONICAL YELPCHI + FROZEN SPLIT GATE
# Add-Input robust version.
#
# No training.
# ============================================================

from pathlib import Path

import hashlib
import io
import json
import shutil
import zipfile

import numpy as np
from scipy.io import loadmat


print(
    "===== STEP 5 — YELPCHI DATASET / SPLIT GATE =====",
    flush=True
)


INPUT_ROOT = Path(
    "/kaggle/input"
)


SHARED_DIR = (
    ROOT /
    "shared" /
    "splits"
)


SHARED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. FROZEN EXPECTED IDENTITIES
# ============================================================

EXPECTED_YELP_SHA = (
    "fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42"
)


EXPECTED_SPLIT_SHA = (
    "0ea0af36dfc5a3a1f381ea2e3168377e45826498b6063190185c673e9ec8c22b"
)


EXPECTED_COUNTS = {

    "TR40":
        18381,

    "TR30":
        13785,

    "TR20":
        9190,

    "TR10":
        4595,

    "val":
        9191,

    "test":
        18382
}


EXPECTED_NODES = 45954

EXPECTED_FEATURES = 32

EXPECTED_FRAUD = 6677

EXPECTED_NORMAL = 39277


# ============================================================
# 2. HASH HELPERS
# ============================================================

def sha256_file(
    path
):

    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:

        for block in iter(
            lambda:
            f.read(
                8 * 1024 * 1024
            ),
            b""
        ):

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(
    data
):

    return hashlib.sha256(
        data
    ).hexdigest()


# ============================================================
# 3. RESOLVE CANONICAL YELPCHI BY SHA256
# ============================================================

print(
    "\n[1/6] Resolving canonical YelpChi...",
    flush=True
)


mat_candidates = [

    p
    for p in INPUT_ROOT.rglob(
        "*.mat"
    )
    if p.is_file()

]


print(
    "MAT candidates:",
    len(
        mat_candidates
    ),
    flush=True
)


YELP_PATH = None


for p in mat_candidates:

    actual = sha256_file(
        p
    )

    print(
        p,
        actual,
        flush=True
    )


    if actual == EXPECTED_YELP_SHA:

        YELP_PATH = p

        break


assert YELP_PATH is not None, (
    "Canonical YelpChi.mat was not found by SHA256. "
    "Do not continue."
)


print(
    "Canonical YelpChi:",
    YELP_PATH,
    flush=True
)

print(
    "[1/6] PASS",
    flush=True
)


# ============================================================
# 4. RESOLVE FROZEN YELP SPLIT
#
# First try directly mounted NPZ files.
# If Kaggle kept the bundle zipped, inspect ZIP members.
# ============================================================

print(
    "\n[2/6] Resolving frozen YelpChi split...",
    flush=True
)


SPLIT_PATH = None


npz_candidates = [

    p
    for p in INPUT_ROOT.rglob(
        "*.npz"
    )
    if p.is_file()

]


for p in npz_candidates:

    actual = sha256_file(
        p
    )


    if actual == EXPECTED_SPLIT_SHA:

        destination = (
            SHARED_DIR /
            "yelp_seed2_nested_splits.npz"
        )


        shutil.copy2(
            p,
            destination
        )


        SPLIT_PATH = destination

        print(
            "Matched direct NPZ:",
            p,
            flush=True
        )

        break


# ------------------------------------------------------------
# ZIP fallback
# ------------------------------------------------------------

if SPLIT_PATH is None:

    zip_candidates = [

        p
        for p in INPUT_ROOT.rglob(
            "*.zip"
        )
        if p.is_file()

    ]


    for zip_path in zip_candidates:

        try:

            with zipfile.ZipFile(
                zip_path
            ) as z:

                for member in z.namelist():

                    if not member.endswith(
                        ".npz"
                    ):

                        continue


                    data = z.read(
                        member
                    )


                    actual = sha256_bytes(
                        data
                    )


                    if (
                        actual
                        == EXPECTED_SPLIT_SHA
                    ):

                        destination = (
                            SHARED_DIR /
                            "yelp_seed2_nested_splits.npz"
                        )


                        destination.write_bytes(
                            data
                        )


                        SPLIT_PATH = (
                            destination
                        )


                        print(
                            "Matched ZIP:",
                            zip_path,
                            flush=True
                        )

                        print(
                            "Member:",
                            member,
                            flush=True
                        )

                        break


            if SPLIT_PATH is not None:

                break


        except zipfile.BadZipFile:

            continue


assert SPLIT_PATH is not None, (
    "Frozen YelpChi split was not found by SHA256."
)


actual_split_sha = sha256_file(
    SPLIT_PATH
)


assert (
    actual_split_sha
    == EXPECTED_SPLIT_SHA
)


print(
    "Frozen split:",
    SPLIT_PATH,
    flush=True
)

print(
    "SHA256:",
    actual_split_sha,
    flush=True
)

print(
    "[2/6] PASS",
    flush=True
)


# ============================================================
# 5. VERIFY SPLIT STRUCTURE
# ============================================================

print(
    "\n[3/6] Verifying frozen split structure...",
    flush=True
)


split = np.load(
    SPLIT_PATH,
    allow_pickle=False
)


required_keys = {

    "TR40",
    "TR30",
    "TR20",
    "TR10",
    "val",
    "test",
    "seed",
    "source_nodes"
}


assert required_keys.issubset(
    set(
        split.files
    )
), (
    "Frozen split is missing required arrays."
)


for key, expected in EXPECTED_COUNTS.items():

    actual = int(
        split[
            key
        ].shape[
            0
        ]
    )

    print(
        key,
        actual,
        flush=True
    )

    assert actual == expected, (
        f"{key} count mismatch: "
        f"{actual} != {expected}"
    )


assert (
    int(
        split[
            "seed"
        ][
            0
        ]
    )
    == 2
)


assert (
    int(
        split[
            "source_nodes"
        ][
            0
        ]
    )
    == EXPECTED_NODES
)


print(
    "[3/6] PASS",
    flush=True
)


# ============================================================
# 6. VERIFY NESTING / DISJOINTNESS / RANGE
# ============================================================

print(
    "\n[4/6] Verifying nesting and isolation...",
    flush=True
)


TR40 = set(
    split[
        "TR40"
    ].tolist()
)

TR30 = set(
    split[
        "TR30"
    ].tolist()
)

TR20 = set(
    split[
        "TR20"
    ].tolist()
)

TR10 = set(
    split[
        "TR10"
    ].tolist()
)

VAL = set(
    split[
        "val"
    ].tolist()
)

TEST = set(
    split[
        "test"
    ].tolist()
)


assert TR10 < TR20

assert TR20 < TR30

assert TR30 < TR40


assert TR40.isdisjoint(
    VAL
)

assert TR40.isdisjoint(
    TEST
)

assert VAL.isdisjoint(
    TEST
)


all_ids = (

    TR40
    | VAL
    | TEST

)


assert min(
    all_ids
) >= 0


assert max(
    all_ids
) < EXPECTED_NODES


assert len(
    all_ids
) == EXPECTED_NODES


print(
    "TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40: YES",
    flush=True
)

print(
    "Validation fixed/disjoint: YES",
    flush=True
)

print(
    "Test fixed/disjoint: YES",
    flush=True
)

print(
    "TR40 + validation + test cover all nodes: YES",
    flush=True
)

print(
    "[4/6] PASS",
    flush=True
)


# ============================================================
# 7. VERIFY CANONICAL YELP CONTENT
# ============================================================

print(
    "\n[5/6] Verifying YelpChi structure...",
    flush=True
)


mat = loadmat(
    YELP_PATH
)


visible_keys = sorted(
    k
    for k in mat.keys()
    if not k.startswith(
        "__"
    )
)


print(
    "MAT keys:",
    visible_keys,
    flush=True
)


assert "features" in mat

assert "label" in mat


features = mat[
    "features"
]


labels = np.asarray(
    mat[
        "label"
    ]
).reshape(
    -1
)


assert (
    features.shape[
        0
    ]
    == EXPECTED_NODES
)


assert (
    features.shape[
        1
    ]
    == EXPECTED_FEATURES
)


assert (
    labels.shape[
        0
    ]
    == EXPECTED_NODES
)


# Normal/fraud labels should be binary.
unique_labels, label_counts = np.unique(
    labels,
    return_counts=True
)


label_summary = {

    int(
        k
    ):
        int(
            v
        )

    for k, v
    in zip(
        unique_labels,
        label_counts
    )
}


print(
    "Nodes:",
    features.shape[
        0
    ],
    flush=True
)

print(
    "Features:",
    features.shape[
        1
    ],
    flush=True
)

print(
    "Label counts:",
    label_summary,
    flush=True
)


assert set(
    label_summary
) == {
    0,
    1
}


assert (
    label_summary[
        1
    ]
    == EXPECTED_FRAUD
)


assert (
    label_summary[
        0
    ]
    == EXPECTED_NORMAL
)


# ------------------------------------------------------------
# YelpChi source relation matrices.
# We accept only the three expected semantic relations.
# ------------------------------------------------------------

relation_keys = [

    key
    for key in [
        "net_rur",
        "net_rtr",
        "net_rsr"
    ]
    if key in mat

]


print(
    "Semantic relation matrices found:",
    relation_keys,
    flush=True
)


assert len(
    relation_keys
) == 3, (
    "Expected the three canonical YelpChi relation matrices."
)


print(
    "[5/6] PASS",
    flush=True
)


# ============================================================
# 8. SPLIT LABEL COUNTS — AUDIT ONLY
# ============================================================

print(
    "\n[6/6] Auditing class composition by split...",
    flush=True
)


composition = {}


for key in [

    "TR40",
    "TR30",
    "TR20",
    "TR10",
    "val",
    "test"

]:

    ids = split[
        key
    ].astype(
        np.int64
    )


    y = labels[
        ids
    ]


    normal = int(
        (
            y == 0
        ).sum()
    )


    fraud = int(
        (
            y == 1
        ).sum()
    )


    composition[
        key
    ] = {

        "nodes":
            int(
                len(
                    ids
                )
            ),

        "normal":
            normal,

        "fraud":
            fraud,

        "fraud_pct":
            (
                100.0
                * fraud
                / len(
                    ids
                )
            )
    }


    print(
        f"{key:<4} "
        f"nodes={len(ids):>6} "
        f"normal={normal:>6} "
        f"fraud={fraud:>5} "
        f"fraud%="
        f"{100.0 * fraud / len(ids):.4f}",
        flush=True
    )


assert all(
    v[
        "fraud"
    ] > 0
    for v in composition.values()
)


# ============================================================
# 9. SAVE DATASET GATE EVIDENCE
# ============================================================

record = {

    "dataset":
        "YelpChi",

    "canonical_path":
        str(
            YELP_PATH
        ),

    "canonical_sha256":
        EXPECTED_YELP_SHA,

    "split_path":
        str(
            SPLIT_PATH
        ),

    "split_sha256":
        EXPECTED_SPLIT_SHA,

    "split_seed":
        2,

    "source_nodes":
        EXPECTED_NODES,

    "feature_dim":
        EXPECTED_FEATURES,

    "normal":
        EXPECTED_NORMAL,

    "fraud":
        EXPECTED_FRAUD,

    "semantic_relations":
        relation_keys,

    "split_counts":
        EXPECTED_COUNTS,

    "split_label_composition":
        composition,

    "nesting_verified":
        True,

    "validation_test_fixed":
        True,

    "training_started":
        False
}


record_path = (
    DIRS[
        "evidence"
    ]
    /
    "step5_yelpchi_dataset_gate.json"
)


record_path.write_text(
    json.dumps(
        record,
        indent=2
    )
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n===== STEP 5 GATE =====",
    flush=True
)

print(
    "PASS — canonical YelpChi and frozen project split verified.",
    flush=True
)

print(
    "YelpChi SHA256:",
    EXPECTED_YELP_SHA,
    flush=True
)

print(
    "Split SHA256:",
    EXPECTED_SPLIT_SHA,
    flush=True
)

print(
    "Nodes: 45,954",
    flush=True
)

print(
    "Features: 32",
    flush=True
)

print(
    "Fraud / normal: 6,677 / 39,277",
    flush=True
)

print(
    "Semantic relations: 3/3",
    flush=True
)

print(
    "Split seed: 2",
    flush=True
)

print(
    "TR10⊂TR20⊂TR30⊂TR40: YES",
    flush=True
)

print(
    "Validation/test fixed and disjoint: YES",
    flush=True
)

print(
    "PMP training started: NO",
    flush=True
)

print(
    "Ready for Step 6 PMP YelpChi adapter audit: YES",
    flush=True
)

===== STEP 5 — YELPCHI DATASET / SPLIT GATE =====

[1/6] Resolving canonical YelpChi...
MAT candidates: 1
/kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42
Canonical YelpChi: /kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat
[1/6] PASS

[2/6] Resolving frozen YelpChi split...
Matched direct NPZ: /kaggle/input/datasets/pathikahmed0007/pmp-frozen-splits/splits/yelp_seed2_nested_splits.npz
Frozen split: /kaggle/working/comp8851_pmp/shared/splits/yelp_seed2_nested_splits.npz
SHA256: 0ea0af36dfc5a3a1f381ea2e3168377e45826498b6063190185c673e9ec8c22b
[2/6] PASS

[3/6] Verifying frozen split structure...
TR40 18381
TR30 13785
TR20 9190
TR10 4595
val 9191
test 18382
[3/6] PASS

[4/6] Verifying nesting and isolation...
TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40: YES
Validation fixed/disjoint: YES
Test fixed/disjoint: YES
TR40 + validation + test cover all nodes: YES
[4/6] PASS

[5/6] Verifying YelpChi structure...
MAT keys: ['fea

## Step 6A — PMP × YelpChi Frozen-Split Adapter Audit

Step 5 verified the canonical YelpChi graph and the exact project split IDs.

The official PMP Yelp implementation already provides the required
three-relation message-passing architecture. This step does not alter that
architecture.

### Controlled adaptation

The unified benchmark must not use the split masks generated internally by
DGL/PMP. A small runtime adapter therefore:

- keeps the canonical YelpChi graph and all three relations;
- keeps PMP feature preprocessing and model code unchanged;
- replaces only train/validation/test masks with the frozen project IDs;
- rebuilds PMP's `label_unk` field so true labels are visible only for the
  selected training nodes;
- rebuilds the DGL loaders from those frozen IDs;
- uses `num_workers=0` as a Kaggle loader-stability setting;
- leaves the official PMP repository itself unmodified.

### Audit conditions

For TR40:

- train IDs must exactly equal frozen `TR40`;
- validation IDs must exactly equal frozen `val`;
- test IDs must exactly equal frozen `test`;
- `label_unk[train]` must equal the true labels;
- every non-training node must have `label_unk = 2`;
- all three YelpChi relations must remain present;
- no model training or test evaluation occurs.

This is a representation/protocol adapter only, not an architecture change.

In [9]:
# ============================================================
# STEP 6A — PMP × YELPCHI FROZEN-SPLIT ADAPTER AUDIT
#
# No training.
# No test evaluation.
# Official PMP source remains unchanged.
# ============================================================

from pathlib import Path

import hashlib
import json
import os
import shutil
import subprocess
import textwrap
import time


print(
    "===== STEP 6A — PMP YELPCHI ADAPTER AUDIT =====",
    flush=True
)


# ============================================================
# 0. REQUIRE PREVIOUS GATES
# ============================================================

assert "ROOT" in globals(), (
    "ROOT missing — Step 1 state is unavailable."
)

assert "DIRS" in globals(), (
    "DIRS missing — Step 1 state is unavailable."
)

assert "YELP_PATH" in globals(), (
    "YELP_PATH missing — rerun Step 5 only."
)

assert "SPLIT_PATH" in globals(), (
    "SPLIT_PATH missing — rerun Step 5 only."
)


LAUNCHER = (
    ROOT /
    "pmp_python.sh"
)


assert LAUNCHER.exists(), (
    "PMP launcher missing — Step 4A/4B environment "
    "must remain available."
)


# ============================================================
# 1. RESOLVE FROZEN PMP REPOSITORY
# ============================================================

SOURCE_ROOT = (
    ROOT /
    "source"
)


repo_candidates = []


for main_file in SOURCE_ROOT.rglob(
    "main.py"
):

    candidate = (
        main_file.parent
    )

    if (
        (
            candidate /
            "config" /
            "yelp.yml"
        ).exists()
        and
        (
            candidate /
            "DataHelper" /
            "datasetHelper.py"
        ).exists()
        and
        (
            candidate /
            ".git"
        ).exists()
    ):

        repo_candidates.append(
            candidate
        )


assert len(
    repo_candidates
) == 1, (
    "Could not resolve exactly one frozen PMP repository. "
    f"Candidates: {repo_candidates}"
)


REPO = (
    repo_candidates[
        0
    ]
)


commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD"
    ],
    text=True
).strip()


EXPECTED_COMMIT = (
    "3f7629f6c180891a0bc1bba3c66d94d288a1ddae"
)


assert commit == EXPECTED_COMMIT


status_before = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--porcelain"
    ],
    text=True
)


assert status_before.strip() == "", (
    "Official PMP repository is not clean before adapter audit."
)


print(
    "Repository:",
    REPO,
    flush=True
)

print(
    "Commit:",
    commit,
    flush=True
)

print(
    "Official source clean: YES",
    flush=True
)


# ============================================================
# 2. PREPARE LOCAL CANONICAL DGL RAW DIRECTORY
#
# DGL FraudDataset expects:
#
#   raw_dir/yelp/YelpChi.mat
#
# We link the canonical Kaggle input instead of downloading
# another copy.
# ============================================================

RAW_ROOT = (
    ROOT /
    "shared" /
    "pmp_raw"
)


YELP_RAW_DIR = (
    RAW_ROOT /
    "yelp"
)


YELP_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)


LOCAL_YELP = (
    YELP_RAW_DIR /
    "YelpChi.mat"
)


if (
    LOCAL_YELP.exists()
    or
    LOCAL_YELP.is_symlink()
):

    LOCAL_YELP.unlink()


LOCAL_YELP.symlink_to(
    Path(
        YELP_PATH
    ).resolve()
)


assert LOCAL_YELP.exists()


print(
    "Canonical local raw link:",
    LOCAL_YELP,
    flush=True
)


# ============================================================
# 3. CREATE RUNTIME UNIFIED-SPLIT ADAPTER
# ============================================================

ADAPTER_DIR = (
    ROOT /
    "adapters"
)


ADAPTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ADAPTER_PATH = (
    ADAPTER_DIR /
    "pmp_unified_split_adapter.py"
)


adapter_code = r'''
from pathlib import Path

import numpy as np
import torch

from dgl.dataloading import (
    DataLoader as DGLDataLoader,
    MultiLayerFullNeighborSampler,
    NeighborSampler,
)


def _mask_from_ids(num_nodes, ids):
    mask = torch.zeros(
        num_nodes,
        dtype=torch.bool
    )

    mask[
        torch.as_tensor(
            ids,
            dtype=torch.long
        )
    ] = True

    return mask


def apply_frozen_split(
    helper,
    split_path,
    ratio="TR40",
    num_workers=0
):
    """
    Replace only PMP/DGL split masks and loaders.

    PMP architecture, graph structure, features, labels,
    relation definitions and message-passing code remain
    unchanged.
    """

    split_path = Path(
        split_path
    )

    split = np.load(
        split_path,
        allow_pickle=False
    )


    assert ratio in {
        "TR40",
        "TR30",
        "TR20",
        "TR10"
    }


    train_ids = (
        split[
            ratio
        ]
        .astype(
            np.int64
        )
    )

    val_ids = (
        split[
            "val"
        ]
        .astype(
            np.int64
        )
    )

    test_ids = (
        split[
            "test"
        ]
        .astype(
            np.int64
        )
    )


    data = helper.data

    num_nodes = int(
        helper.num_nodes
    )


    train_mask = _mask_from_ids(
        num_nodes,
        train_ids
    )

    val_mask = _mask_from_ids(
        num_nodes,
        val_ids
    )

    test_mask = _mask_from_ids(
        num_nodes,
        test_ids
    )


    # --------------------------------------------------------
    # Replace only the split masks.
    # --------------------------------------------------------

    data.ndata[
        "train_mask"
    ] = train_mask

    data.ndata[
        "val_mask"
    ] = val_mask

    data.ndata[
        "test_mask"
    ] = test_mask


    # Re-run PMP's own metadata configuration using the
    # unchanged graph and dataset object.
    helper.config_data(
        data,
        helper.dataset
    )


    # --------------------------------------------------------
    # PMP mechanism:
    #
    # true labels only for training nodes;
    # every other node remains unknown (=2).
    # --------------------------------------------------------

    label_unk = torch.full(
        (
            helper.num_nodes,
        ),
        2,
        dtype=torch.long
    )


    label_unk[
        helper.train_nid
    ] = helper.labels[
        helper.train_nid
    ]


    helper.data.ndata[
        "label_unk"
    ] = label_unk


    # --------------------------------------------------------
    # Rebuild the sampler exactly from PMP config.
    # --------------------------------------------------------

    sampled_neighbors = (
        helper.config[
            "sampled_neighbors"
        ]
    )


    if helper.config.get(
        "full_neighbors",
        False
    ):

        sampler = (
            MultiLayerFullNeighborSampler(
                num_layers=helper.config[
                    "n_layer"
                ]
            )
        )

    else:

        full_relations = (
            helper.data.canonical_etypes
        )


        train_fanouts = []


        for i in range(
            helper.config[
                "n_layer"
            ]
        ):

            train_fanouts.append(
                {
                    etype:
                        sampled_neighbors[
                            i
                        ]

                    for etype
                    in full_relations
                }
            )


        sampler = NeighborSampler(
            (
                train_fanouts
                if not helper.config[
                    "homo"
                ]
                else sampled_neighbors
            ),
            prefetch_node_feats=[
                "feature"
            ],
            prefetch_labels=[
                "label"
            ]
        )


    # --------------------------------------------------------
    # Controlled Kaggle loader policy.
    #
    # num_workers changes only CPU loader execution, not PMP
    # architecture or graph/data semantics.
    # --------------------------------------------------------

    nw = int(
        num_workers
    )


    helper.train_loader = DGLDataLoader(
        helper.data,
        helper.train_nid,
        sampler,
        batch_size=helper.config[
            "batch_size"
        ],
        shuffle=True,
        drop_last=False,
        num_workers=nw
    )


    helper.val_loader = DGLDataLoader(
        helper.data,
        helper.val_nid,
        sampler,
        batch_size=helper.config.get(
            "val_batch_size",
            helper.config[
                "batch_size"
            ]
        ),
        shuffle=False,
        drop_last=False,
        num_workers=nw
    )


    helper.test_loader = DGLDataLoader(
        helper.data,
        helper.test_nid,
        sampler,
        batch_size=helper.config.get(
            "test_batch_size",
            helper.config[
                "batch_size"
            ]
        ),
        shuffle=False,
        drop_last=False,
        num_workers=nw
    )


    return helper
'''


ADAPTER_PATH.write_text(
    textwrap.dedent(
        adapter_code
    ).lstrip(),
    encoding="utf-8"
)


def sha256_file(
    path
):

    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:

        for block in iter(
            lambda:
                f.read(
                    8 * 1024 * 1024
                ),
            b""
        ):

            h.update(
                block
            )

    return h.hexdigest()


adapter_sha = sha256_file(
    ADAPTER_PATH
)


print(
    "Adapter:",
    ADAPTER_PATH,
    flush=True
)

print(
    "Adapter SHA256:",
    adapter_sha,
    flush=True
)


# ============================================================
# 4. WRITE CHILD AUDIT SCRIPT
#
# Run under isolated PMP Python 3.10, not notebook Python 3.12.
# ============================================================

AUDIT_SCRIPT = (
    DIRS[
        "evidence"
    ] /
    "step6a_yelp_adapter_audit.py"
)


AUDIT_JSON = (
    DIRS[
        "evidence"
    ] /
    "step6a_yelp_adapter_audit.json"
)


audit_code = r'''
import json
import sys
from pathlib import Path

import numpy as np
import torch
import yaml


repo = Path(
    sys.argv[1]
)

adapter_dir = Path(
    sys.argv[2]
)

raw_root = Path(
    sys.argv[3]
)

split_path = Path(
    sys.argv[4]
)

output_json = Path(
    sys.argv[5]
)


sys.path.insert(
    0,
    str(repo)
)

sys.path.insert(
    0,
    str(adapter_dir)
)


from DataHelper.datasetHelper import DatasetHelper
from pmp_unified_split_adapter import apply_frozen_split


# ------------------------------------------------------------
# Load official frozen Yelp configuration.
# ------------------------------------------------------------

with (
    repo /
    "config" /
    "yelp.yml"
).open(
    "r"
) as f:

    raw_cfg = yaml.safe_load(
        f
    )


cfg = dict(
    raw_cfg[
        "LA-SAGE-S"
    ]
)


# Fields normally injected by main.py / args2config.
cfg[
    "model_name"
] = "LA-SAGE-S"

cfg[
    "model"
] = "LA-SAGE-S"

cfg[
    "dataset"
] = "yelp"

cfg[
    "gpu_id"
] = 0

cfg[
    "train_size"
] = 0.4

cfg[
    "val_size"
] = 0.2

cfg[
    "num_workers"
] = 0


# Unified-mode selection rules.
cfg[
    "monitor"
] = "ap_gnn"

cfg[
    "test_each_epoch"
] = False


# ------------------------------------------------------------
# Load graph through PMP's official DatasetHelper.
# ------------------------------------------------------------

helper = DatasetHelper(
    cfg,
    dName="yelp"
)


helper.dataset_source_folder_path = str(
    raw_root
)


helper.load()


# ------------------------------------------------------------
# Replace author-generated masks with frozen project IDs.
# ------------------------------------------------------------

helper = apply_frozen_split(
    helper,
    split_path,
    ratio="TR40",
    num_workers=0
)


split = np.load(
    split_path,
    allow_pickle=False
)


def sorted_np(
    tensor_or_array
):

    if torch.is_tensor(
        tensor_or_array
    ):

        x = (
            tensor_or_array
            .detach()
            .cpu()
            .numpy()
        )

    else:

        x = np.asarray(
            tensor_or_array
        )

    return np.sort(
        x.astype(
            np.int64
        )
    )


# ------------------------------------------------------------
# Exact split identity.
# ------------------------------------------------------------

assert np.array_equal(
    sorted_np(
        helper.train_nid
    ),
    np.sort(
        split[
            "TR40"
        ]
    )
)


assert np.array_equal(
    sorted_np(
        helper.val_nid
    ),
    np.sort(
        split[
            "val"
        ]
    )
)


assert np.array_equal(
    sorted_np(
        helper.test_nid
    ),
    np.sort(
        split[
            "test"
        ]
    )
)


# ------------------------------------------------------------
# Graph / relation identity.
# ------------------------------------------------------------

expected_relations = {
    "net_rsr",
    "net_rtr",
    "net_rur"
}


actual_relations = set(
    helper.relations
)


assert actual_relations == expected_relations


assert int(
    helper.num_nodes
) == 45954


assert int(
    helper.feat_dim
) == 32


assert int(
    (
        helper.labels
        == 1
    ).sum()
) == 6677


assert int(
    (
        helper.labels
        == 0
    ).sum()
) == 39277


# ------------------------------------------------------------
# Train-only PMP partition-label audit.
# ------------------------------------------------------------

label_unk = (
    helper.data.ndata[
        "label_unk"
    ]
    .detach()
    .cpu()
)


labels = (
    helper.labels
    .detach()
    .cpu()
)


train_nid = (
    helper.train_nid
    .detach()
    .cpu()
)


assert torch.equal(
    label_unk[
        train_nid
    ],
    labels[
        train_nid
    ]
)


outside_train = torch.ones(
    helper.num_nodes,
    dtype=torch.bool
)


outside_train[
    train_nid
] = False


assert torch.all(
    label_unk[
        outside_train
    ]
    == 2
)


train_label_values = sorted(
    set(
        label_unk[
            train_nid
        ].tolist()
    )
)


outside_label_values = sorted(
    set(
        label_unk[
            outside_train
        ].tolist()
    )
)


assert train_label_values == [
    0,
    1
]


assert outside_label_values == [
    2
]


# ------------------------------------------------------------
# Loaders exist but are NOT iterated in Step 6A.
# ------------------------------------------------------------

assert helper.train_loader is not None
assert helper.val_loader is not None
assert helper.test_loader is not None


record = {

    "dataset":
        "YelpChi",

    "mode":
        "UNIFIED ADAPTER AUDIT",

    "ratio":
        "TR40",

    "nodes":
        int(
            helper.num_nodes
        ),

    "feature_dim":
        int(
            helper.feat_dim
        ),

    "relations":
        sorted(
            actual_relations
        ),

    "num_relations":
        int(
            helper.num_relations
        ),

    "train_nodes":
        int(
            len(
                helper.train_nid
            )
        ),

    "val_nodes":
        int(
            len(
                helper.val_nid
            )
        ),

    "test_nodes":
        int(
            len(
                helper.test_nid
            )
        ),

    "monitor":
        cfg[
            "monitor"
        ],

    "test_each_epoch":
        bool(
            cfg[
                "test_each_epoch"
            ]
        ),

    "full_neighbors":
        bool(
            cfg[
                "full_neighbors"
            ]
        ),

    "batch_size":
        int(
            cfg[
                "batch_size"
            ]
        ),

    "loader_num_workers":
        0,

    "train_label_unk_values":
        train_label_values,

    "outside_train_label_unk_values":
        outside_label_values,

    "train_only_partition_labels_verified":
        True,

    "test_loader_iterated":
        False,

    "model_initialized":
        False,

    "training_performed":
        False
}


output_json.write_text(
    json.dumps(
        record,
        indent=2
    )
)


print(
    "===== CHILD ADAPTER AUDIT ====="
)

print(
    "Nodes:",
    helper.num_nodes
)

print(
    "Stored relation-wise edges:",
    helper.data.num_edges()
)

print(
    "Relations:",
    sorted(
        actual_relations
    )
)

print(
    "TR40 / VAL / TEST:",
    len(
        helper.train_nid
    ),
    len(
        helper.val_nid
    ),
    len(
        helper.test_nid
    )
)

print(
    "Training label_unk values:",
    train_label_values
)

print(
    "Non-training label_unk values:",
    outside_label_values
)

print(
    "Validation monitor:",
    cfg[
        "monitor"
    ]
)

print(
    "test_each_epoch:",
    cfg[
        "test_each_epoch"
    ]
)

print(
    "Training performed: NO"
)

print(
    "Test loader iterated: NO"
)
'''


AUDIT_SCRIPT.write_text(
    textwrap.dedent(
        audit_code
    ).lstrip(),
    encoding="utf-8"
)


# ============================================================
# 5. RUN CHILD AUDIT
# ============================================================

AUDIT_LOG = (
    DIRS[
        "logs"
    ] /
    "step6a_yelp_adapter_audit.log"
)


cmd = [

    str(
        LAUNCHER
    ),

    str(
        AUDIT_SCRIPT
    ),

    str(
        REPO
    ),

    str(
        ADAPTER_DIR
    ),

    str(
        RAW_ROOT
    ),

    str(
        SPLIT_PATH
    ),

    str(
        AUDIT_JSON
    )
]


print(
    "\n$",
    " ".join(
        cmd
    ),
    flush=True
)


p = subprocess.run(
    cmd,
    cwd=str(
        REPO
    ),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=1200,
    check=False
)


AUDIT_LOG.write_text(
    p.stdout,
    encoding="utf-8"
)


print(
    p.stdout,
    flush=True
)


assert p.returncode == 0, (
    "Step 6A child audit failed. "
    f"See {AUDIT_LOG}"
)


assert AUDIT_JSON.exists()


audit = json.loads(
    AUDIT_JSON.read_text()
)


assert audit[
    "training_performed"
] is False


assert audit[
    "test_loader_iterated"
] is False


assert audit[
    "train_only_partition_labels_verified"
] is True


# ============================================================
# 6. CONFIRM OFFICIAL REPO STILL UNMODIFIED
# ============================================================

status_after = subprocess.check_output(
    [
        "git",
        "-C",
        str(
            REPO
        ),
        "status",
        "--porcelain"
    ],
    text=True
)


assert status_after.strip() == "", (
    "Official PMP source was modified unexpectedly."
)


SOURCE_PATCH_RECORD = (
    DIRS[
        "patches"
    ] /
    "step6a_official_source_changes.txt"
)


SOURCE_PATCH_RECORD.write_text(
    "NO UPSTREAM PMP SOURCE MODIFICATION\n"
    f"Frozen commit: {EXPECTED_COMMIT}\n"
    "Unified split handling is implemented in a separate "
    "runtime adapter.\n",
    encoding="utf-8"
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n===== STEP 6A GATE =====",
    flush=True
)

print(
    "PASS — PMP YelpChi frozen-split adapter verified.",
    flush=True
)

print(
    "Frozen TR40 IDs used: YES",
    flush=True
)

print(
    "Frozen validation IDs used: YES",
    flush=True
)

print(
    "Frozen test IDs registered: YES",
    flush=True
)

print(
    "Semantic relations preserved: 3/3",
    flush=True
)

print(
    "Training labels exposed to PMP partition mechanism: YES",
    flush=True
)

print(
    "Validation/test labels exposed to partition mechanism: NO",
    flush=True
)

print(
    "Validation selection metric prepared: AUPRC",
    flush=True
)

print(
    "Test-each-epoch disabled: YES",
    flush=True
)

print(
    "Official PMP source modified: NO",
    flush=True
)

print(
    "Architecture modified: NO",
    flush=True
)

print(
    "Training performed: NO",
    flush=True
)

print(
    "Ready for Step 6B one-epoch smoke: YES",
    flush=True
)

===== STEP 6A — PMP YELPCHI ADAPTER AUDIT =====
Repository: /kaggle/working/comp8851_pmp/source/PMP
Commit: 3f7629f6c180891a0bc1bba3c66d94d288a1ddae
Official source clean: YES
Canonical local raw link: /kaggle/working/comp8851_pmp/shared/pmp_raw/yelp/YelpChi.mat
Adapter: /kaggle/working/comp8851_pmp/adapters/pmp_unified_split_adapter.py
Adapter SHA256: e35ec052058c0c7f92352c8692ad86b7a62eccc9584e86a3fa23708afea88956

$ /kaggle/working/comp8851_pmp/pmp_python.sh /kaggle/working/comp8851_pmp/evidence/step6a_yelp_adapter_audit.py /kaggle/working/comp8851_pmp/source/PMP /kaggle/working/comp8851_pmp/adapters /kaggle/working/comp8851_pmp/shared/pmp_raw /kaggle/working/comp8851_pmp/shared/splits/yelp_seed2_nested_splits.npz /kaggle/working/comp8851_pmp/evidence/step6a_yelp_adapter_audit.json
Extracting file to /kaggle/working/comp8851_pmp/shared/pmp_raw/yelp_a7a80596
Done saving data into cached files.
[Global] Dataset <yelp> Overview
	 Num Edges 8051348
	 Num Features     32
	Entire (fraud/t

AssertionError: Official PMP source was modified unexpectedly.

## Step 6A.1 — Source-Cleanliness and DGL Dataset-Identity Recovery

The Step 6A child audit completed successfully, but the final repository
cleanliness assertion detected runtime files created while importing the PMP
Python modules.

Runtime artefacts such as `__pycache__` and `.pyc` files are not modifications
to the PMP implementation.

A second issue was observed during Step 6A: DGL downloaded/extracted its own
FraudYelp archive rather than directly consuming the canonical Kaggle symlink.

Before any training, this recovery gate therefore:

1. prints the exact Git status that caused the previous assertion;
2. distinguishes tracked source changes from untracked Python runtime artefacts;
3. removes only safe Python cache artefacts;
4. refuses to delete or ignore any unknown repository file;
5. hashes every `YelpChi.mat` materialized under the PMP/DGL raw-data directory;
6. confirms every materialized YelpChi copy is byte-for-byte identical to the
   canonical dataset verified in Step 5;
7. confirms the official PMP tracked source remains unchanged.

No model training or test evaluation occurs.

In [10]:
# ============================================================
# STEP 6A.1 — SOURCE CLEANLINESS + DGL DATA IDENTITY RECOVERY
#
# No training.
# No test evaluation.
# ============================================================

from pathlib import Path

import hashlib
import json
import shutil
import subprocess


print(
    "===== STEP 6A.1 — PMP POST-AUDIT RECOVERY =====",
    flush=True
)


# ============================================================
# 0. REQUIRED STATE
# ============================================================

assert "REPO" in globals()
assert "RAW_ROOT" in globals()
assert "YELP_PATH" in globals()
assert "EXPECTED_YELP_SHA" in globals()
assert "DIRS" in globals()


EXPECTED_COMMIT = (
    "3f7629f6c180891a0bc1bba3c66d94d288a1ddae"
)


def git_status():

    return subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO),
            "status",
            "--porcelain=v1",
            "-uall"
        ],
        text=True
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for block in iter(
            lambda:
                f.read(
                    8 * 1024 * 1024
                ),
            b""
        ):

            h.update(
                block
            )

    return h.hexdigest()


# ============================================================
# 1. VERIFY FROZEN COMMIT
# ============================================================

commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD"
    ],
    text=True
).strip()


assert commit == EXPECTED_COMMIT


print(
    "Frozen commit:",
    commit,
    flush=True
)


# ============================================================
# 2. SHOW EXACT STATUS THAT CAUSED STEP 6A FAILURE
# ============================================================

status_before = git_status()


print(
    "\n===== GIT STATUS BEFORE SAFE CLEANUP =====",
    flush=True
)


if status_before.strip():

    print(
        status_before,
        flush=True
    )

else:

    print(
        "(clean)",
        flush=True
    )


status_lines = [

    line
    for line in status_before.splitlines()
    if line.strip()

]


tracked_changes_before = [

    line
    for line in status_lines
    if not line.startswith("??")
]


untracked_before = [

    line
    for line in status_lines
    if line.startswith("??")
]


print(
    "Tracked change entries:",
    len(
        tracked_changes_before
    ),
    flush=True
)

print(
    "Untracked entries:",
    len(
        untracked_before
    ),
    flush=True
)


# A tracked change is a real source-state problem.
assert not tracked_changes_before, (
    "A TRACKED PMP repository file changed. "
    "Do not continue. Send the Git status output."
)


# ============================================================
# 3. REMOVE ONLY SAFE PYTHON RUNTIME ARTEFACTS
# ============================================================

print(
    "\n===== SAFE RUNTIME CLEANUP =====",
    flush=True
)


removed = []


# ------------------------------------------------------------
# __pycache__ directories
# ------------------------------------------------------------

for p in sorted(
    REPO.rglob(
        "__pycache__"
    )
):

    if p.is_dir():

        removed.append(
            str(
                p.relative_to(
                    REPO
                )
            )
        )

        shutil.rmtree(
            p
        )


# ------------------------------------------------------------
# .pytest_cache directories
# ------------------------------------------------------------

for p in sorted(
    REPO.rglob(
        ".pytest_cache"
    )
):

    if p.is_dir():

        removed.append(
            str(
                p.relative_to(
                    REPO
                )
            )
        )

        shutil.rmtree(
            p
        )


# ------------------------------------------------------------
# Standalone Python bytecode files
# ------------------------------------------------------------

for pattern in [
    "*.pyc",
    "*.pyo"
]:

    for p in sorted(
        REPO.rglob(
            pattern
        )
    ):

        if p.is_file():

            removed.append(
                str(
                    p.relative_to(
                        REPO
                    )
                )
            )

            p.unlink()


print(
    "Safe runtime artefacts removed:",
    len(
        removed
    ),
    flush=True
)


for item in removed[
    :50
]:

    print(
        "  removed:",
        item,
        flush=True
    )


if len(
    removed
) > 50:

    print(
        "  ...",
        flush=True
    )


# ============================================================
# 4. RECHECK REPOSITORY
# ============================================================

status_after = git_status()


print(
    "\n===== GIT STATUS AFTER SAFE CLEANUP =====",
    flush=True
)


if status_after.strip():

    print(
        status_after,
        flush=True
    )

else:

    print(
        "(clean)",
        flush=True
    )


remaining_lines = [

    line
    for line in status_after.splitlines()
    if line.strip()

]


assert not remaining_lines, (
    "Repository still contains unexpected files after "
    "safe cache cleanup. Nothing else was deleted. "
    "STOP and send the remaining Git status."
)


# ------------------------------------------------------------
# Strong tracked-file verification.
# ------------------------------------------------------------

tracked_diff = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--quiet"
    ],
    check=False
)


staged_diff = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--cached",
        "--quiet"
    ],
    check=False
)


assert tracked_diff.returncode == 0

assert staged_diff.returncode == 0


print(
    "Tracked PMP source changed: NO",
    flush=True
)


# ============================================================
# 5. VERIFY ORIGINAL CANONICAL KAGGLE INPUT AGAIN
# ============================================================

canonical_sha = sha256_file(
    YELP_PATH
)


assert (
    canonical_sha
    == EXPECTED_YELP_SHA
)


print(
    "\nCanonical Kaggle YelpChi SHA256:",
    canonical_sha,
    flush=True
)


# ============================================================
# 6. FIND EVERY YELPCHI.MAT MATERIALIZED BY DGL
# ============================================================

raw_yelp_files = sorted(
    {
        p.resolve()
        for p in Path(
            RAW_ROOT
        ).rglob(
            "YelpChi.mat"
        )
        if p.exists()
    }
)


print(
    "\n===== DGL / PMP YELPCHI MATERIALIZATIONS =====",
    flush=True
)

print(
    "YelpChi.mat files found:",
    len(
        raw_yelp_files
    ),
    flush=True
)


assert raw_yelp_files, (
    "No YelpChi.mat exists under PMP raw-data storage."
)


raw_records = []


for p in raw_yelp_files:

    actual_sha = sha256_file(
        p
    )


    same = (
        actual_sha
        == EXPECTED_YELP_SHA
    )


    record = {

        "path":
            str(
                p
            ),

        "sha256":
            actual_sha,

        "canonical_match":
            same
    }


    raw_records.append(
        record
    )


    print(
        p,
        flush=True
    )

    print(
        "  SHA256:",
        actual_sha,
        flush=True
    )

    print(
        "  canonical match:",
        "YES"
        if same
        else "NO",
        flush=True
    )


# Every YelpChi raw copy DGL/PMP can currently see must be
# canonical. We do not silently accept another dataset version.
assert all(
    x[
        "canonical_match"
    ]
    for x in raw_records
), (
    "At least one DGL-materialized YelpChi.mat differs from "
    "the canonical Step 5 dataset. Do not train."
)


# ============================================================
# 7. VERIFY STEP 6A CHILD EVIDENCE
# ============================================================

AUDIT_JSON = (
    DIRS[
        "evidence"
    ] /
    "step6a_yelp_adapter_audit.json"
)


assert AUDIT_JSON.exists()


audit = json.loads(
    AUDIT_JSON.read_text()
)


assert audit[
    "train_only_partition_labels_verified"
] is True

assert audit[
    "test_loader_iterated"
] is False

assert audit[
    "training_performed"
] is False

assert sorted(
    audit[
        "relations"
    ]
) == [
    "net_rsr",
    "net_rtr",
    "net_rur"
]


# ============================================================
# 8. SAVE RECOVERY RECORD
# ============================================================

recovery_record = {

    "model":
        "PMP",

    "dataset":
        "YelpChi",

    "step":
        "6A.1",

    "frozen_commit":
        commit,

    "status_before_cleanup":
        status_before.splitlines(),

    "tracked_changes_before_cleanup":
        tracked_changes_before,

    "untracked_before_cleanup":
        untracked_before,

    "safe_runtime_artifacts_removed":
        removed,

    "status_after_cleanup":
        status_after.splitlines(),

    "tracked_source_modified":
        False,

    "canonical_kaggle_yelp_sha256":
        canonical_sha,

    "expected_yelp_sha256":
        EXPECTED_YELP_SHA,

    "dgl_raw_yelp_materializations":
        raw_records,

    "all_dgl_yelp_materializations_canonical":
        True,

    "step6a_train_only_partition_labels_verified":
        True,

    "step6a_test_loader_iterated":
        False,

    "training_performed":
        False
}


RECOVERY_JSON = (
    DIRS[
        "evidence"
    ] /
    "step6a1_post_audit_recovery.json"
)


RECOVERY_JSON.write_text(
    json.dumps(
        recovery_record,
        indent=2
    )
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n===== STEP 6A.1 GATE =====",
    flush=True
)

print(
    "PASS — Step 6A audit recovered and strengthened.",
    flush=True
)

print(
    "Official PMP tracked source modified: NO",
    flush=True
)

print(
    "Runtime Python cache residue removed: YES",
    flush=True
)

print(
    "Unknown repository residue remaining: NO",
    flush=True
)

print(
    "Canonical Kaggle YelpChi reverified: YES",
    flush=True
)

print(
    "All DGL-materialized YelpChi copies canonical: YES",
    flush=True
)

print(
    "Frozen TR40/validation/test audit retained: YES",
    flush=True
)

print(
    "PMP train-only partition labels retained: YES",
    flush=True
)

print(
    "Test evaluation performed: NO",
    flush=True
)

print(
    "Training performed: NO",
    flush=True
)

print(
    "Ready for Step 6B one-epoch smoke: YES",
    flush=True
)

===== STEP 6A.1 — PMP POST-AUDIT RECOVERY =====
Frozen commit: 3f7629f6c180891a0bc1bba3c66d94d288a1ddae

===== GIT STATUS BEFORE SAFE CLEANUP =====
?? DataHelper/__pycache__/dataset.cpython-310.pyc
?? DataHelper/__pycache__/datasetHelper.cpython-310.pyc

Tracked change entries: 0
Untracked entries: 2

===== SAFE RUNTIME CLEANUP =====
Safe runtime artefacts removed: 1
  removed: DataHelper/__pycache__

===== GIT STATUS AFTER SAFE CLEANUP =====
(clean)
Tracked PMP source changed: NO

Canonical Kaggle YelpChi SHA256: fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42

===== DGL / PMP YELPCHI MATERIALIZATIONS =====
YelpChi.mat files found: 2
/kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat
  SHA256: fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42
  canonical match: YES
/kaggle/working/comp8851_pmp/shared/pmp_raw/yelp_a7a80596/YelpChi.mat
  SHA256: fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42
  canonical match: YES

===== STEP 6A

## Step 6A.2 — Exact Frozen PMP Training API Audit

The first API-audit attempt found an important implementation detail before
training:

`Trainer` at the frozen PMP commit does not expose a method named `init`.

No model initialization, training, validation iteration or test evaluation
occurred during that failed introspection attempt.

This corrected audit makes no assumptions about method names.

It dynamically records:

- every method actually defined on `Trainer`;
- each method's exact signature and source location;
- the complete `Trainer` class source;
- how `Trainer` is instantiated and called by the frozen `main.py`;
- relevant training/evaluation/checkpoint functions in the repository;
- the official YelpChi `LA-SAGE-S` configuration;
- locations of AUPRC/AP, AUROC, threshold, validation and test-control code.

The result of this gate will be used to construct the one-epoch smoke exactly
around the real PMP API rather than adapting PMP to an assumed interface.

No training or test evaluation is performed.

In [12]:
# ============================================================
# STEP 6A.2 — DYNAMIC PMP TRAINING API AUDIT
#
# Corrected version:
# assumes NO Trainer method names.
#
# No model initialization.
# No DataLoader iteration.
# No training.
# No validation/test evaluation.
# ============================================================

from pathlib import Path

import json
import shutil
import subprocess
import textwrap


print(
    "===== STEP 6A.2 — DYNAMIC PMP TRAINING API AUDIT =====",
    flush=True
)


# ============================================================
# 0. REQUIRE PREVIOUS STATE
# ============================================================

assert "REPO" in globals()
assert "LAUNCHER" in globals()
assert "DIRS" in globals()


EXPECTED_COMMIT = (
    "3f7629f6c180891a0bc1bba3c66d94d288a1ddae"
)


commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD"
    ],
    text=True
).strip()


assert commit == EXPECTED_COMMIT


# Tracked source must already be unchanged.
assert subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--quiet"
    ],
    check=False
).returncode == 0


assert subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--cached",
        "--quiet"
    ],
    check=False
).returncode == 0


# ============================================================
# 1. OUTPUT FILES
# ============================================================

API_SCRIPT = (
    DIRS["evidence"]
    / "step6a2_dynamic_pmp_api_audit.py"
)


API_JSON = (
    DIRS["evidence"]
    / "step6a2_dynamic_pmp_api_audit.json"
)


API_LOG = (
    DIRS["logs"]
    / "step6a2_dynamic_pmp_api_audit.log"
)


TRAINER_SOURCE_TXT = (
    DIRS["evidence"]
    / "step6a2_trainer_source.txt"
)


MAIN_EXCERPT_TXT = (
    DIRS["evidence"]
    / "step6a2_main_usage_excerpt.txt"
)


# ============================================================
# 2. CHILD SCRIPT
# ============================================================

api_code = r'''
import inspect
import json
import re
import sys

from pathlib import Path

import yaml


repo = Path(
    sys.argv[1]
)

output_json = Path(
    sys.argv[2]
)

trainer_source_txt = Path(
    sys.argv[3]
)

main_excerpt_txt = Path(
    sys.argv[4]
)


sys.path.insert(
    0,
    str(repo)
)


print(
    "===== CHILD DYNAMIC PMP API AUDIT =====",
    flush=True
)


# ============================================================
# 1. IMPORT REAL TRAINER
# ============================================================

import training_procedure


assert hasattr(
    training_procedure,
    "Trainer"
)


Trainer = (
    training_procedure.Trainer
)


print(
    "training_procedure:",
    training_procedure.__file__,
    flush=True
)

print(
    "Trainer:",
    Trainer,
    flush=True
)

print(
    "Trainer source file:",
    inspect.getsourcefile(
        Trainer
    ),
    flush=True
)


# ============================================================
# 2. RECORD EVERY ACTUAL TRAINER METHOD
# ============================================================

method_records = {}


print(
    "\n===== ACTUAL TRAINER METHODS =====",
    flush=True
)


for name, obj in inspect.getmembers(
    Trainer
):

    if name.startswith(
        "__"
    ) and name not in {
        "__init__"
    }:

        continue


    if not callable(
        obj
    ):

        continue


    try:

        signature = str(
            inspect.signature(
                obj
            )
        )

    except Exception:

        signature = (
            "<signature unavailable>"
        )


    try:

        source_file = (
            inspect.getsourcefile(
                obj
            )
        )

    except Exception:

        source_file = None


    try:

        source_lines, start_line = (
            inspect.getsourcelines(
                obj
            )
        )

        preview = "".join(
            source_lines[
                :40
            ]
        )

    except Exception:

        start_line = None
        preview = ""


    method_records[
        name
    ] = {

        "signature":
            signature,

        "source_file":
            source_file,

        "source_start_line":
            start_line,

        "source_preview":
            preview
    }


    print(
        f"{name}{signature}",
        flush=True
    )

    print(
        "  file:",
        source_file,
        flush=True
    )

    print(
        "  line:",
        start_line,
        flush=True
    )


assert len(
    method_records
) > 0


# ============================================================
# 3. SAVE COMPLETE TRAINER CLASS SOURCE
# ============================================================

try:

    trainer_source = inspect.getsource(
        Trainer
    )

except Exception as e:

    trainer_source = (
        "TRAINER SOURCE EXTRACTION FAILED:\n"
        + repr(
            e
        )
    )


trainer_source_txt.write_text(
    trainer_source,
    encoding="utf-8"
)


print(
    "\nTrainer source saved:",
    trainer_source_txt,
    flush=True
)


# ============================================================
# 4. MAIN.PY — FIND HOW TRAINER IS REALLY USED
# ============================================================

MAIN_PATH = (
    repo
    / "main.py"
)


assert MAIN_PATH.exists()


main_lines = MAIN_PATH.read_text(
    encoding="utf-8"
).splitlines()


usage_patterns = [

    r"\bTrainer\b",
    r"\btrainer\b",
    r"\.train\(",
    r"\.evaluation\(",
    r"\.eval",
    r"test_each_epoch",
    r"monitor",
    r"EarlyStopping",
    r"best_",
    r"threshold"
]


matching_line_numbers = set()


for i, line in enumerate(
    main_lines,
    start=1
):

    for pattern in usage_patterns:

        if re.search(
            pattern,
            line,
            flags=re.IGNORECASE
        ):

            # Include local context.
            for j in range(
                max(
                    1,
                    i - 5
                ),
                min(
                    len(
                        main_lines
                    ),
                    i + 5
                ) + 1
            ):

                matching_line_numbers.add(
                    j
                )

            break


main_excerpt_parts = []


for line_no in sorted(
    matching_line_numbers
):

    text = (
        f"{line_no:04d}: "
        f"{main_lines[line_no - 1]}"
    )

    main_excerpt_parts.append(
        text
    )


main_excerpt = "\n".join(
    main_excerpt_parts
)


main_excerpt_txt.write_text(
    main_excerpt,
    encoding="utf-8"
)


print(
    "\n===== MAIN.PY TRAINER USAGE =====",
    flush=True
)

print(
    main_excerpt,
    flush=True
)


# ============================================================
# 5. OFFICIAL YELP CONFIG
# ============================================================

CONFIG_PATH = (
    repo
    / "config"
    / "yelp.yml"
)


with CONFIG_PATH.open(
    "r"
) as f:

    full_cfg = yaml.safe_load(
        f
    )


assert "LA-SAGE-S" in full_cfg


cfg = dict(
    full_cfg[
        "LA-SAGE-S"
    ]
)


print(
    "\n===== OFFICIAL YELP LA-SAGE-S CONFIG =====",
    flush=True
)


for key in sorted(
    cfg.keys()
):

    print(
        f"{key}: {cfg[key]}",
        flush=True
    )


# ============================================================
# 6. FIND RELEVANT FUNCTIONS/CLASSES ACROSS SOURCE
# ============================================================

source_function_records = []


interesting_names = {
    "train",
    "test",
    "evaluate",
    "evaluation",
    "eval_model",
    "validate",
    "validation",
    "forward",
    "run",
    "fit",
    "main",
    "test_model",
    "train_model"
}


print(
    "\n===== RELEVANT SOURCE DEFINITIONS =====",
    flush=True
)


for py_path in sorted(
    repo.rglob(
        "*.py"
    )
):

    # Ignore generated runtime cache paths.
    if "__pycache__" in py_path.parts:

        continue


    try:

        source = py_path.read_text(
            encoding="utf-8"
        )

    except Exception:

        continue


    lines = source.splitlines()


    for i, line in enumerate(
        lines,
        start=1
    ):

        stripped = line.strip()


        match = re.match(
            r"(?:async\s+)?def\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(",
            stripped
        )


        if match:

            fn_name = match.group(
                1
            )


            if (
                fn_name
                in interesting_names
                or
                "train" in fn_name.lower()
                or
                "eval" in fn_name.lower()
                or
                "test" in fn_name.lower()
                or
                "valid" in fn_name.lower()
            ):

                rec = {

                    "file":
                        str(
                            py_path.relative_to(
                                repo
                            )
                        ),

                    "line":
                        i,

                    "definition":
                        stripped
                }


                source_function_records.append(
                    rec
                )


                print(
                    f"{rec['file']}:{i} "
                    f"{stripped}",
                    flush=True
                )


# ============================================================
# 7. METRIC / CHECKPOINT / THRESHOLD TOKEN LOCATIONS
# ============================================================

tokens = [

    "ap_gnn",
    "auc_gnn",
    "f1_macro",
    "recall_1",
    "precision_1",
    "best_pr_thres",

    "test_each_epoch",

    "monitor",

    "early_stop",
    "earlystopping",

    "threshold",
    "thres",

    "save_model",
    "load_model"
]


token_hits = {
    token: []
    for token in tokens
}


for py_path in sorted(
    repo.rglob(
        "*.py"
    )
):

    if "__pycache__" in py_path.parts:

        continue


    try:

        text = py_path.read_text(
            encoding="utf-8"
        )

    except Exception:

        continue


    lines = text.splitlines()


    for token in tokens:

        token_lower = (
            token.lower()
        )


        for i, line in enumerate(
            lines,
            start=1
        ):

            if (
                token_lower
                in line.lower()
            ):

                token_hits[
                    token
                ].append(
                    {
                        "file":
                            str(
                                py_path.relative_to(
                                    repo
                                )
                            ),

                        "line":
                            i,

                        "text":
                            line.strip()
                    }
                )


print(
    "\n===== METRIC / CONTROL TOKEN LOCATIONS =====",
    flush=True
)


for token in tokens:

    hits = token_hits[
        token
    ]


    print(
        f"\n[{token}] "
        f"{len(hits)} hit(s)",
        flush=True
    )


    for hit in hits[
        :15
    ]:

        print(
            f"  {hit['file']}:{hit['line']} "
            f"{hit['text']}",
            flush=True
        )


# ============================================================
# 8. CONSTRUCTOR SIGNATURE
# ============================================================

trainer_constructor_signature = str(
    inspect.signature(
        Trainer
    )
)


print(
    "\n===== TRAINER CONSTRUCTOR =====",
    flush=True
)

print(
    "Trainer",
    trainer_constructor_signature,
    flush=True
)


# ============================================================
# 9. RESULT RECORD
# ============================================================

record = {

    "repository_commit":
        "3f7629f6c180891a0bc1bba3c66d94d288a1ddae",

    "training_procedure_module":
        training_procedure.__file__,

    "trainer_source_file":
        inspect.getsourcefile(
            Trainer
        ),

    "trainer_constructor_signature":
        trainer_constructor_signature,

    "trainer_methods":
        method_records,

    "main_usage_excerpt":
        main_excerpt,

    "official_yelp_config":
        cfg,

    "relevant_source_definitions":
        source_function_records,

    "metric_control_token_hits":
        token_hits,

    "model_initialized":
        False,

    "dataloader_iterated":
        False,

    "training_performed":
        False,

    "validation_evaluation_performed":
        False,

    "test_evaluation_performed":
        False
}


output_json.write_text(
    json.dumps(
        record,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)


print(
    "\n===== CHILD DYNAMIC API AUDIT COMPLETE =====",
    flush=True
)

print(
    "Model initialized: NO",
    flush=True
)

print(
    "DataLoader iterated: NO",
    flush=True
)

print(
    "Training performed: NO",
    flush=True
)

print(
    "Validation evaluated: NO",
    flush=True
)

print(
    "Test evaluated: NO",
    flush=True
)
'''


API_SCRIPT.write_text(
    textwrap.dedent(
        api_code
    ).lstrip(),
    encoding="utf-8"
)


# ============================================================
# 3. RUN CHILD AUDIT
# ============================================================

cmd = [

    str(
        LAUNCHER
    ),

    str(
        API_SCRIPT
    ),

    str(
        REPO
    ),

    str(
        API_JSON
    ),

    str(
        TRAINER_SOURCE_TXT
    ),

    str(
        MAIN_EXCERPT_TXT
    )
]


print(
    "\n$",
    " ".join(
        cmd
    ),
    flush=True
)


p = subprocess.run(
    cmd,
    cwd=str(
        REPO
    ),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=600,
    check=False
)


API_LOG.write_text(
    p.stdout,
    encoding="utf-8"
)


print(
    p.stdout,
    flush=True
)


assert p.returncode == 0, (
    "Corrected Step 6A.2 API audit failed. "
    f"See {API_LOG}"
)


assert API_JSON.exists()


audit = json.loads(
    API_JSON.read_text()
)


# ============================================================
# 4. SAFETY GATES
# ============================================================

assert audit[
    "model_initialized"
] is False


assert audit[
    "dataloader_iterated"
] is False


assert audit[
    "training_performed"
] is False


assert audit[
    "validation_evaluation_performed"
] is False


assert audit[
    "test_evaluation_performed"
] is False


assert len(
    audit[
        "trainer_methods"
    ]
) > 0


# ============================================================
# 5. REMOVE ONLY IMPORT-GENERATED PYTHON CACHE
# ============================================================

for pth in sorted(
    REPO.rglob(
        "__pycache__"
    )
):

    if pth.is_dir():

        shutil.rmtree(
            pth
        )


for pattern in [
    "*.pyc",
    "*.pyo"
]:

    for pth in sorted(
        REPO.rglob(
            pattern
        )
    ):

        if pth.is_file():

            pth.unlink()


# ============================================================
# 6. OFFICIAL SOURCE MUST STILL BE UNCHANGED
# ============================================================

assert subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--quiet"
    ],
    check=False
).returncode == 0


assert subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--cached",
        "--quiet"
    ],
    check=False
).returncode == 0


status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--porcelain=v1",
        "-uall"
    ],
    text=True
)


assert status.strip() == "", (
    "Unexpected repository residue after dynamic API audit:\n"
    + status
)


# ============================================================
# 7. NOTE THE IMPORTANT CORRECTION
# ============================================================

trainer_methods = (
    audit[
        "trainer_methods"
    ]
)


print(
    "\n===== CORRECTED API SUMMARY =====",
    flush=True
)


print(
    "Trainer constructor:",
    audit[
        "trainer_constructor_signature"
    ],
    flush=True
)


print(
    "Actual Trainer methods:",
    ", ".join(
        sorted(
            trainer_methods.keys()
        )
    ),
    flush=True
)


print(
    "Trainer.init exists:",
    (
        "YES"
        if "init" in trainer_methods
        else "NO"
    ),
    flush=True
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n===== STEP 6A.2 GATE =====",
    flush=True
)

print(
    "PASS — exact frozen PMP training API captured dynamically.",
    flush=True
)

print(
    "Trainer method names assumed in advance: NO",
    flush=True
)

print(
    "Trainer constructor captured: YES",
    flush=True
)

print(
    "All actual Trainer methods/signatures captured: YES",
    flush=True
)

print(
    "main.py Trainer usage captured: YES",
    flush=True
)

print(
    "Official Yelp LA-SAGE-S config captured: YES",
    flush=True
)

print(
    "Metric/checkpoint/threshold code locations captured: YES",
    flush=True
)

print(
    "Official tracked source modified: NO",
    flush=True
)

print(
    "Training performed: NO",
    flush=True
)

print(
    "Validation evaluation performed: NO",
    flush=True
)

print(
    "Test evaluation performed: NO",
    flush=True
)

print(
    "Ready to construct exact Step 6B smoke: YES",
    flush=True
)

===== STEP 6A.2 — DYNAMIC PMP TRAINING API AUDIT =====

$ /kaggle/working/comp8851_pmp/pmp_python.sh /kaggle/working/comp8851_pmp/evidence/step6a2_dynamic_pmp_api_audit.py /kaggle/working/comp8851_pmp/source/PMP /kaggle/working/comp8851_pmp/evidence/step6a2_dynamic_pmp_api_audit.json /kaggle/working/comp8851_pmp/evidence/step6a2_trainer_source.txt /kaggle/working/comp8851_pmp/evidence/step6a2_main_usage_excerpt.txt
===== CHILD DYNAMIC PMP API AUDIT =====
training_procedure: /kaggle/working/comp8851_pmp/source/PMP/training_procedure/__init__.py
Trainer: <class 'training_procedure.Trainer'>
Trainer source file: /kaggle/working/comp8851_pmp/source/PMP/training_procedure/__init__.py

===== ACTUAL TRAINER METHODS =====
__init__(self, args, config, logger)
  file: /kaggle/working/comp8851_pmp/source/PMP/training_procedure/__init__.py
  line: 11

Trainer source saved: /kaggle/working/comp8851_pmp/evidence/step6a2_trainer_source.txt

===== MAIN.PY TRAINER USAGE =====
0005: import torch
0006: i

## Step 6A.3 — PMP Runtime Method-Binding Audit

Step 6A.2 confirmed that the frozen `Trainer` class defines only `__init__`,
while `main.py` calls runtime attributes named:

- `init`
- `train`
- `evaluation`
- `eval_model`

This indicates that PMP dynamically binds its training, preparation and
evaluation functions to each `Trainer` instance.

Before the first training epoch, this gate creates a `Trainer` instance only
and records the exact bound runtime methods and their signatures.

The gate does not:

- initialize a model;
- iterate a DataLoader;
- call `init`;
- call `train`;
- call `evaluation`;
- access validation labels through a loader;
- access test data.

The purpose is to construct Step 6B against the exact runtime API exposed by
the frozen PMP commit rather than relying on assumptions.

In [13]:
# ============================================================
# STEP 6A.3 — PMP RUNTIME METHOD-BINDING AUDIT
#
# Trainer instance creation only.
# No model initialization.
# No DataLoader iteration.
# No training.
# No validation/test evaluation.
# ============================================================

from pathlib import Path

import json
import shutil
import subprocess
import textwrap


print(
    "===== STEP 6A.3 — PMP RUNTIME BINDING AUDIT =====",
    flush=True
)


# ============================================================
# 0. REQUIRED STATE
# ============================================================

assert "REPO" in globals()
assert "LAUNCHER" in globals()
assert "DIRS" in globals()


EXPECTED_COMMIT = (
    "3f7629f6c180891a0bc1bba3c66d94d288a1ddae"
)


commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD"
    ],
    text=True
).strip()


assert commit == EXPECTED_COMMIT


# ============================================================
# 1. OUTPUT FILES
# ============================================================

SCRIPT = (
    DIRS["evidence"]
    / "step6a3_runtime_binding_audit.py"
)


OUTPUT_JSON = (
    DIRS["evidence"]
    / "step6a3_runtime_binding_audit.json"
)


LOG = (
    DIRS["logs"]
    / "step6a3_runtime_binding_audit.log"
)


INIT_SOURCE_TXT = (
    DIRS["evidence"]
    / "step6a3_trainer_init_source.txt"
)


# ============================================================
# 2. CHILD SCRIPT
# ============================================================

child_code = r'''
import inspect
import json
import sys

from pathlib import Path
from types import SimpleNamespace

import yaml


repo = Path(
    sys.argv[1]
)

output_json = Path(
    sys.argv[2]
)

init_source_txt = Path(
    sys.argv[3]
)


sys.path.insert(
    0,
    str(repo)
)


print(
    "===== CHILD PMP RUNTIME BINDING AUDIT =====",
    flush=True
)


# ============================================================
# 1. IMPORT TRAINER
# ============================================================

from training_procedure import Trainer


# ============================================================
# 2. CAPTURE TRAINER.__INIT__ SOURCE
# ============================================================

init_source = inspect.getsource(
    Trainer.__init__
)


init_source_txt.write_text(
    init_source,
    encoding="utf-8"
)


print(
    "\n===== TRAINER.__INIT__ SOURCE =====",
    flush=True
)

print(
    init_source,
    flush=True
)


# ============================================================
# 3. OFFICIAL YELP CONFIG
# ============================================================

with (
    repo
    / "config"
    / "yelp.yml"
).open(
    "r"
) as f:

    all_cfg = yaml.safe_load(
        f
    )


cfg = dict(
    all_cfg[
        "LA-SAGE-S"
    ]
)


# Only values that main.py normally injects and that may be
# needed while constructing Trainer.
cfg[
    "model_name"
] = "LA-SAGE-S"

cfg[
    "model"
] = "LA-SAGE-S"

cfg[
    "dataset"
] = "yelp"

cfg[
    "gpu_id"
] = 0


# Unified control values that do not initialize a model.
cfg[
    "monitor"
] = "ap_gnn"

cfg[
    "test_each_epoch"
] = False

cfg[
    "num_workers"
] = 0


# ============================================================
# 4. MINIMAL MAIN-LIKE ARGS
# ============================================================

args = SimpleNamespace(
    gpu_id=0,
    seed=2,
    num_workers=0,
    dataset="yelp",
    train_size=0.4,
    val_size=0.2
)


# ============================================================
# 5. MINIMAL LOGGER
#
# Only supplied because Trainer constructor expects one.
# Nothing is written outside this audit.
# ============================================================

class AuditLogger:

    def log(
        self,
        *args,
        **kwargs
    ):
        pass

    def add_line(
        self,
        *args,
        **kwargs
    ):
        pass


logger = AuditLogger()


# ============================================================
# 6. CONSTRUCT TRAINER INSTANCE ONLY
# ============================================================

T = Trainer(
    config=cfg,
    args=args,
    logger=logger
)


print(
    "\nTrainer instance created: YES",
    flush=True
)


# ============================================================
# 7. ENUMERATE INSTANCE ATTRIBUTES
# ============================================================

instance_callables = {}


print(
    "\n===== INSTANCE CALLABLE ATTRIBUTES =====",
    flush=True
)


for name in sorted(
    dir(
        T
    )
):

    if name.startswith(
        "__"
    ):

        continue


    try:

        obj = getattr(
            T,
            name
        )

    except Exception:

        continue


    if not callable(
        obj
    ):

        continue


    try:

        signature = str(
            inspect.signature(
                obj
            )
        )

    except Exception:

        signature = (
            "<signature unavailable>"
        )


    try:

        source_file = (
            inspect.getsourcefile(
                obj
            )
        )

    except Exception:

        source_file = None


    try:

        source_lines, start_line = (
            inspect.getsourcelines(
                obj
            )
        )

        preview = "".join(
            source_lines[
                :35
            ]
        )

    except Exception:

        start_line = None
        preview = ""


    instance_callables[
        name
    ] = {

        "signature":
            signature,

        "source_file":
            source_file,

        "source_start_line":
            start_line,

        "source_preview":
            preview
    }


    print(
        f"{name}{signature}",
        flush=True
    )

    print(
        "  source:",
        source_file,
        flush=True
    )

    print(
        "  line:",
        start_line,
        flush=True
    )


# ============================================================
# 8. REQUIRED PMP RUNTIME METHODS
# ============================================================

required = [
    "init",
    "train",
    "evaluation",
    "eval_model"
]


print(
    "\n===== REQUIRED RUNTIME METHODS =====",
    flush=True
)


for name in required:

    exists = hasattr(
        T,
        name
    )


    print(
        name,
        "->",
        "FOUND"
        if exists
        else "MISSING",
        flush=True
    )


    assert exists, (
        f"Trainer instance does not expose {name}"
    )


# ============================================================
# 9. PRINT EXACT REQUIRED SIGNATURES
# ============================================================

print(
    "\n===== EXACT BOUND METHOD SIGNATURES =====",
    flush=True
)


for name in required:

    obj = getattr(
        T,
        name
    )


    print(
        f"{name}: "
        f"{inspect.signature(obj)}",
        flush=True
    )


# ============================================================
# 10. SAVE RECORD
# ============================================================

record = {

    "repository_commit":
        "3f7629f6c180891a0bc1bba3c66d94d288a1ddae",

    "trainer_instance_created":
        True,

    "runtime_callables":
        instance_callables,

    "required_runtime_methods":
        required,

    "required_runtime_methods_present":
        all(
            hasattr(
                T,
                x
            )
            for x in required
        ),

    "model_initialized":
        False,

    "dataloader_iterated":
        False,

    "training_performed":
        False,

    "validation_evaluation_performed":
        False,

    "test_evaluation_performed":
        False
}


output_json.write_text(
    json.dumps(
        record,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)


print(
    "\n===== CHILD BINDING AUDIT COMPLETE =====",
    flush=True
)

print(
    "Trainer instance created: YES",
    flush=True
)

print(
    "Model initialized: NO",
    flush=True
)

print(
    "DataLoader iterated: NO",
    flush=True
)

print(
    "Training performed: NO",
    flush=True
)

print(
    "Validation evaluated: NO",
    flush=True
)

print(
    "Test evaluated: NO",
    flush=True
)
'''


SCRIPT.write_text(
    textwrap.dedent(
        child_code
    ).lstrip(),
    encoding="utf-8"
)


# ============================================================
# 3. RUN CHILD SCRIPT
# ============================================================

cmd = [

    str(
        LAUNCHER
    ),

    str(
        SCRIPT
    ),

    str(
        REPO
    ),

    str(
        OUTPUT_JSON
    ),

    str(
        INIT_SOURCE_TXT
    )
]


print(
    "\n$",
    " ".join(
        cmd
    ),
    flush=True
)


p = subprocess.run(
    cmd,
    cwd=str(
        REPO
    ),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=600,
    check=False
)


LOG.write_text(
    p.stdout,
    encoding="utf-8"
)


print(
    p.stdout,
    flush=True
)


assert p.returncode == 0, (
    "Step 6A.3 runtime-binding audit failed. "
    f"See {LOG}"
)


assert OUTPUT_JSON.exists()


audit = json.loads(
    OUTPUT_JSON.read_text()
)


# ============================================================
# 4. SAFETY ASSERTIONS
# ============================================================

assert audit[
    "trainer_instance_created"
] is True


assert audit[
    "required_runtime_methods_present"
] is True


assert audit[
    "model_initialized"
] is False


assert audit[
    "dataloader_iterated"
] is False


assert audit[
    "training_performed"
] is False


assert audit[
    "validation_evaluation_performed"
] is False


assert audit[
    "test_evaluation_performed"
] is False


# ============================================================
# 5. CLEAN ONLY IMPORT-GENERATED PYTHON CACHE
# ============================================================

for pth in sorted(
    REPO.rglob(
        "__pycache__"
    )
):

    if pth.is_dir():

        shutil.rmtree(
            pth
        )


for pattern in [
    "*.pyc",
    "*.pyo"
]:

    for pth in sorted(
        REPO.rglob(
            pattern
        )
    ):

        if pth.is_file():

            pth.unlink()


# ============================================================
# 6. OFFICIAL TRACKED SOURCE MUST STILL BE UNCHANGED
# ============================================================

assert subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--quiet"
    ],
    check=False
).returncode == 0


assert subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--cached",
        "--quiet"
    ],
    check=False
).returncode == 0


status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--porcelain=v1",
        "-uall"
    ],
    text=True
)


assert status.strip() == "", (
    "Unexpected repository residue after runtime audit:\n"
    + status
)


# ============================================================
# FINAL GATE
# ============================================================

runtime = audit[
    "runtime_callables"
]


print(
    "\n===== STEP 6A.3 GATE =====",
    flush=True
)

print(
    "PASS — PMP runtime method binding verified.",
    flush=True
)

print(
    "Trainer instance created: YES",
    flush=True
)

print(
    "Runtime init method: FOUND",
    flush=True
)

print(
    "Runtime train method: FOUND",
    flush=True
)

print(
    "Runtime evaluation method: FOUND",
    flush=True
)

print(
    "Runtime eval_model method: FOUND",
    flush=True
)

print(
    "init signature:",
    runtime[
        "init"
    ][
        "signature"
    ],
    flush=True
)

print(
    "train signature:",
    runtime[
        "train"
    ][
        "signature"
    ],
    flush=True
)

print(
    "evaluation signature:",
    runtime[
        "evaluation"
    ][
        "signature"
    ],
    flush=True
)

print(
    "eval_model signature:",
    runtime[
        "eval_model"
    ][
        "signature"
    ],
    flush=True
)

print(
    "Model initialized: NO",
    flush=True
)

print(
    "Training performed: NO",
    flush=True
)

print(
    "Test evaluation performed: NO",
    flush=True
)

print(
    "Official tracked source modified: NO",
    flush=True
)

print(
    "Ready for exact Step 6B one-epoch smoke: YES",
    flush=True
)

===== STEP 6A.3 — PMP RUNTIME BINDING AUDIT =====

$ /kaggle/working/comp8851_pmp/pmp_python.sh /kaggle/working/comp8851_pmp/evidence/step6a3_runtime_binding_audit.py /kaggle/working/comp8851_pmp/source/PMP /kaggle/working/comp8851_pmp/evidence/step6a3_runtime_binding_audit.json /kaggle/working/comp8851_pmp/evidence/step6a3_trainer_init_source.txt
===== CHILD PMP RUNTIME BINDING AUDIT =====

===== TRAINER.__INIT__ SOURCE =====
    def __init__(self, args, config, logger):
        self.config = config
        self.logger = logger
        self.args = args
        self.flags = {}
        self.split_info = None

        
        self.prepare_train = partial(prepare_train, self)
        self.prepare_model = partial(prepare_model, self)
        # self.prepare_sage = partial(prepare_sage, self)
        # self.load_data     = partial(load_data, self)
        self.init          = partial(init, self)

        self.train 		   = partial(train, self)
        self.evaluation    = partial(evaluate, s

## Step 6B — PMP × YelpChi TR40 One-Epoch Unified Smoke

The previous gates established the complete controlled training path:

- canonical YelpChi bytes are verified;
- the exact frozen TR40/validation/test IDs are verified;
- all three YelpChi relations are preserved;
- PMP's partition mechanism receives true labels only for training nodes;
- validation/test partition labels remain unknown;
- the official PMP source is unchanged;
- the exact runtime Trainer API is now known.

This step performs the first actual controlled PMP training operation.

### Smoke configuration

- Model: PMP / LA-SAGE-S
- Dataset: YelpChi
- Unified ratio: TR40
- Split seed: 2
- Training seed: 2
- Epochs: 1 only
- Base architecture/configuration: official Yelp LA-SAGE-S settings
- GPU: controlled visible T4 GPU 0 only
- Loader workers after unified adaptation: 0
- Test evaluation: disabled

### Validation

Validation is performed after the one training epoch.

For smoke diagnostics only, the validation predictions use threshold 0.5.

The cell records:

- training loss;
- training epoch time;
- validation inference time;
- validation AUPRC;
- validation AUROC;
- validation Macro-F1 at 0.5;
- validation fraud precision/recall at 0.5;
- parameter count;
- peak training GPU memory;
- peak validation GPU memory.

The threshold is not selected or frozen in this smoke.

The test loader is registered by the dataset adapter but is never iterated.

No test metric is produced.

In [ ]:
# ============================================================
# STEP 6B — PMP × YELPCHI TR40 ONE-EPOCH UNIFIED SMOKE
#
# Actual training: 1 epoch
# Validation: YES
# Test evaluation: NO
# Checkpoint selection: NO
# ============================================================

from pathlib import Path

import json
import shutil
import subprocess
import textwrap


print(
    "===== STEP 6B — PMP × YELPCHI TR40 SMOKE =====",
    flush=True
)


# ============================================================
# 0. REQUIRE PREVIOUS GATES
# ============================================================

required_globals = [

    "ROOT",
    "DIRS",
    "REPO",
    "LAUNCHER",
    "ADAPTER_DIR",
    "RAW_ROOT",
    "SPLIT_PATH"
]


for name in required_globals:

    assert name in globals(), (
        f"{name} missing. "
        "Do not rerun from Step 1; restore the immediately "
        "preceding notebook state."
    )


assert (
    DIRS["evidence"]
    / "step6a3_runtime_binding_audit.json"
).exists()


# ============================================================
# 1. OUTPUT LOCATIONS
# ============================================================

SMOKE_DIR = (

    DIRS["results_unified"]
    / "yelp"
    / "smoke"
)


SMOKE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


SMOKE_SCRIPT = (

    DIRS["evidence"]
    / "step6b_yelp_tr40_smoke.py"
)


SMOKE_JSON = (

    SMOKE_DIR
    / "yelp_tr40_seed2_smoke.json"
)


SMOKE_CONFIG = (

    SMOKE_DIR
    / "yelp_tr40_seed2_smoke_config.yml"
)


SMOKE_LOG = (

    DIRS["logs"]
    / "step6b_yelp_tr40_smoke.log"
)


# ============================================================
# 2. CHILD TRAINING SCRIPT
# ============================================================

smoke_code = r'''
import json
import random
import sys
import time

from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import yaml

from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)


repo = Path(
    sys.argv[1]
)

adapter_dir = Path(
    sys.argv[2]
)

raw_root = Path(
    sys.argv[3]
)

split_path = Path(
    sys.argv[4]
)

output_json = Path(
    sys.argv[5]
)

output_config = Path(
    sys.argv[6]
)


sys.path.insert(
    0,
    str(repo)
)

sys.path.insert(
    0,
    str(adapter_dir)
)


from DataHelper.datasetHelper import DatasetHelper
from training_procedure import Trainer

from pmp_unified_split_adapter import (
    apply_frozen_split
)


# ============================================================
# 1. CONTROLLED DEVICE
# ============================================================

assert torch.cuda.is_available(), (
    "CUDA unavailable inside PMP environment."
)


assert torch.cuda.device_count() == 1, (
    "Controlled PMP launcher must expose exactly one GPU."
)


torch.cuda.set_device(
    0
)


GPU_NAME = torch.cuda.get_device_name(
    0
)


assert "T4" in GPU_NAME.upper(), (
    f"Controlled GPU is not a T4: {GPU_NAME}"
)


DEVICE = torch.device(
    "cuda:0"
)


print(
    "Controlled GPU:",
    GPU_NAME,
    flush=True
)

print(
    "Visible CUDA devices:",
    torch.cuda.device_count(),
    flush=True
)


# ============================================================
# 2. CONTROLLED TRAINING SEED
# ============================================================

TRAIN_SEED = 2


random.seed(
    TRAIN_SEED
)

np.random.seed(
    TRAIN_SEED
)

torch.manual_seed(
    TRAIN_SEED
)

torch.cuda.manual_seed_all(
    TRAIN_SEED
)


# Keep deterministic settings explicit where supported.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


print(
    "Training seed:",
    TRAIN_SEED,
    flush=True
)


# ============================================================
# 3. LOAD OFFICIAL FROZEN YELP CONFIG
# ============================================================

CONFIG_PATH = (

    repo
    / "config"
    / "yelp.yml"
)


with CONFIG_PATH.open(
    "r"
) as f:

    config_root = yaml.safe_load(
        f
    )


assert "LA-SAGE-S" in config_root


cfg = dict(
    config_root[
        "LA-SAGE-S"
    ]
)


# ============================================================
# 4. MAIN.PY-INJECTED VALUES
# ============================================================

cfg[
    "model_name"
] = "LA-SAGE-S"

cfg[
    "model"
] = "LA-SAGE-S"

cfg[
    "dataset"
] = "yelp"

cfg[
    "gpu_id"
] = 0

cfg[
    "seed"
] = TRAIN_SEED


# ============================================================
# 5. UNIFIED-PROTOCOL CONTROLS
# ============================================================

cfg[
    "train_size"
] = 0.4

cfg[
    "val_size"
] = 0.2


# Loader-only Kaggle compatibility setting.
cfg[
    "num_workers"
] = 0


# Later tuning selects checkpoints using validation AUPRC.
cfg[
    "monitor"
] = "ap_gnn"


# Critical test-isolation control.
cfg[
    "test_each_epoch"
] = False


# Smoke only.
cfg[
    "epochs"
] = 1


# We do not perform checkpoint/early-stop selection in smoke.
cfg[
    "patience"
] = 0


# Threshold 0.5 is diagnostic only.
cfg[
    "threshold_moving"
] = True

cfg[
    "thres"
] = 0.5


# Prevent any checkpoint path from being created inside
# the frozen official repository.
cfg[
    "best_model_path"
] = str(
    output_json.parent
    / "unused_smoke_checkpoint.pth"
)


# Save the exact effective smoke config.
output_config.write_text(
    yaml.safe_dump(
        cfg,
        sort_keys=True
    ),
    encoding="utf-8"
)


# ============================================================
# 6. MAIN-LIKE ARGUMENT OBJECT
# ============================================================

args = SimpleNamespace(

    gpu_id=0,

    seed=TRAIN_SEED,

    dataset="yelp",

    num_workers=0,

    train_size=0.4,

    val_size=0.2,

    multirun=1,

    run_best=False,

    data_dir=str(
        raw_root
    ),

    best_model_path=str(
        output_json.parent
    )
)


# ============================================================
# 7. NULL LOGGER
#
# Trainer receives a logger in the official code, but this
# smoke does not require author logging.
# ============================================================

class NullLogger:

    def __init__(
        self
    ):

        self.append = ""


    def log(
        self,
        *args,
        **kwargs
    ):

        pass


    def add_line(
        self,
        *args,
        **kwargs
    ):

        pass


logger = NullLogger()


# ============================================================
# 8. LOAD CANONICAL YELP THROUGH PMP
# ============================================================

helper = DatasetHelper(
    cfg,
    dName="yelp"
)


helper.dataset_source_folder_path = str(
    raw_root
)


helper.load()


# ============================================================
# 9. APPLY EXACT FROZEN PROJECT SPLIT
# ============================================================

helper = apply_frozen_split(

    helper,

    split_path,

    ratio="TR40",

    num_workers=0
)


assert len(
    helper.train_nid
) == 18381


assert len(
    helper.val_nid
) == 9191


assert len(
    helper.test_nid
) == 18382


print(
    "TR40 / VAL / TEST:",
    len(
        helper.train_nid
    ),
    len(
        helper.val_nid
    ),
    len(
        helper.test_nid
    ),
    flush=True
)


# ============================================================
# 10. REVERIFY PMP TRAIN-ONLY PARTITION LABELS
# ============================================================

label_unk = (

    helper.data.ndata[
        "label_unk"
    ]
    .detach()
    .cpu()
)


true_labels = (

    helper.labels
    .detach()
    .cpu()
)


train_ids_cpu = (

    helper.train_nid
    .detach()
    .cpu()
)


assert torch.equal(

    label_unk[
        train_ids_cpu
    ],

    true_labels[
        train_ids_cpu
    ]
)


outside_train = torch.ones(

    helper.num_nodes,

    dtype=torch.bool
)


outside_train[
    train_ids_cpu
] = False


assert torch.all(

    label_unk[
        outside_train
    ]
    == 2
)


print(
    "PMP train-only partition labels: VERIFIED",
    flush=True
)


# ============================================================
# 11. CREATE TRAINER INSTANCE
# ============================================================

T = Trainer(

    config=cfg,

    args=args,

    logger=logger
)


# Exact runtime API already audited.
for method in [

    "init",
    "train",
    "evaluation",
    "eval_model"

]:

    assert hasattr(
        T,
        method
    )


# ============================================================
# 12. INITIALIZE ORIGINAL PMP MODEL
# ============================================================

model, optimizer, loss_func, scheduler = (

    T.init(
        helper
    )
)


assert model is not None

assert optimizer is not None

assert loss_func is not None


# ============================================================
# 13. OPTIMIZER AUDIT
#
# Project controlled Adam defaults:
# beta1=.9, beta2=.999, eps=1e-8
# ============================================================

optimizer_name = type(
    optimizer
).__name__


print(
    "Optimizer:",
    optimizer_name,
    flush=True
)


assert optimizer_name.lower() == "adam"


param_group = (
    optimizer.param_groups[
        0
    ]
)


optimizer_betas = tuple(
    param_group[
        "betas"
    ]
)


optimizer_eps = float(
    param_group[
        "eps"
    ]
)


optimizer_lr = float(
    param_group[
        "lr"
    ]
)


optimizer_weight_decay = float(
    param_group[
        "weight_decay"
    ]
)


assert optimizer_betas == (
    0.9,
    0.999
)


assert abs(
    optimizer_eps
    - 1e-8
) < 1e-15


print(
    "Adam betas:",
    optimizer_betas,
    flush=True
)

print(
    "Adam eps:",
    optimizer_eps,
    flush=True
)

print(
    "Learning rate:",
    optimizer_lr,
    flush=True
)

print(
    "Weight decay:",
    optimizer_weight_decay,
    flush=True
)


# ============================================================
# 14. MODEL PARAMETER COUNT
# ============================================================

parameter_count = int(

    sum(

        p.numel()

        for p
        in model.parameters()
    )
)


trainable_parameter_count = int(

    sum(

        p.numel()

        for p
        in model.parameters()

        if p.requires_grad
    )
)


print(
    "Parameters:",
    parameter_count,
    flush=True
)

print(
    "Trainable parameters:",
    trainable_parameter_count,
    flush=True
)


# ============================================================
# 15. ONE TRAINING EPOCH
#
# Timing excludes model initialization and validation.
# ============================================================

torch.cuda.empty_cache()


torch.cuda.reset_peak_memory_stats(
    0
)


torch.cuda.synchronize(
    0
)


train_start = time.perf_counter()


model, epoch_loss = T.train(

    0,

    model,

    loss_func,

    optimizer,

    helper.train_loader,

    helper
)


torch.cuda.synchronize(
    0
)


train_seconds = (

    time.perf_counter()
    - train_start
)


peak_train_memory_mb = float(

    torch.cuda.max_memory_allocated(
        0
    )

    / (
        1024 ** 2
    )
)


# ============================================================
# 16. NORMALIZE TRAIN LOSS
# ============================================================

if torch.is_tensor(
    epoch_loss
):

    total_train_loss = float(

        epoch_loss
        .detach()
        .cpu()
        .item()
    )

else:

    total_train_loss = float(
        epoch_loss
    )


num_train_batches = int(
    len(
        helper.train_loader
    )
)


assert num_train_batches > 0


average_train_loss = (

    total_train_loss
    / num_train_batches
)


print(
    "\n===== TRAINING COMPLETE =====",
    flush=True
)

print(
    "Train batches:",
    num_train_batches,
    flush=True
)

print(
    "Total train loss:",
    total_train_loss,
    flush=True
)

print(
    "Average train loss:",
    average_train_loss,
    flush=True
)

print(
    "Training seconds:",
    f"{train_seconds:.6f}",
    flush=True
)

print(
    "Peak training GPU memory:",
    f"{peak_train_memory_mb:.2f} MB",
    flush=True
)


# ============================================================
# 17. VALIDATION ONLY
#
# TEST LOADER IS NOT ITERATED.
# ============================================================

torch.cuda.reset_peak_memory_stats(
    0
)


torch.cuda.synchronize(
    0
)


val_start = time.perf_counter()


val_labels, val_probs, val_preds = (

    T.evaluation(

        helper,

        helper.val_loader,

        model,

        threshold_moving=True,

        thres=0.5
    )
)


torch.cuda.synchronize(
    0
)


val_seconds = (

    time.perf_counter()
    - val_start
)


peak_val_memory_mb = float(

    torch.cuda.max_memory_allocated(
        0
    )

    / (
        1024 ** 2
    )
)


# ============================================================
# 18. NORMALIZE VALIDATION OUTPUTS
# ============================================================

def to_numpy(
    x
):

    if torch.is_tensor(
        x
    ):

        return (

            x
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        x
    )


y_val = to_numpy(
    val_labels
).reshape(
    -1
)


p_val = to_numpy(
    val_probs
).reshape(
    -1
)


pred_val = to_numpy(
    val_preds
).reshape(
    -1
).astype(
    np.int64
)


assert len(
    y_val
) == 9191


assert len(
    p_val
) == 9191


assert len(
    pred_val
) == 9191


assert np.isfinite(
    p_val
).all()


# ============================================================
# 19. COMMON SMOKE METRICS
# ============================================================

common_val_auprc = float(

    average_precision_score(
        y_val,
        p_val
    )
)


common_val_auroc = float(

    roc_auc_score(
        y_val,
        p_val
    )
)


common_val_macro_f1 = float(

    f1_score(
        y_val,
        pred_val,
        average="macro"
    )
)


common_val_fraud_precision = float(

    precision_score(
        y_val,
        pred_val,
        pos_label=1,
        zero_division=0
    )
)


common_val_fraud_recall = float(

    recall_score(
        y_val,
        pred_val,
        pos_label=1,
        zero_division=0
    )
)


# ============================================================
# 20. PMP NATIVE EVALUATOR CROSS-CHECK
#
# This remains validation only.
# ============================================================

pmp_val_results = T.eval_model(

    y_val,

    p_val,

    pred_val
)


pmp_val_auprc = float(
    pmp_val_results.ap_gnn
)


pmp_val_auroc = float(
    pmp_val_results.auc_gnn
)


# Ranking metrics must agree with the common calculations.
assert abs(
    pmp_val_auprc
    - common_val_auprc
) < 1e-10


assert abs(
    pmp_val_auroc
    - common_val_auroc
) < 1e-10


# ============================================================
# 21. SMOKE METRIC SANITY
# ============================================================

for value in [

    common_val_auprc,
    common_val_auroc,
    common_val_macro_f1,
    common_val_fraud_precision,
    common_val_fraud_recall

]:

    assert (
        0.0
        <= value
        <= 1.0
    )


assert train_seconds > 0

assert val_seconds > 0

assert peak_train_memory_mb > 0

assert parameter_count > 0


# ============================================================
# 22. SAVE SMOKE RECORD
# ============================================================

record = {

    "model":
        "PMP",

    "repository_model":
        "LA-SAGE-S",

    "repository_commit":
        "3f7629f6c180891a0bc1bba3c66d94d288a1ddae",

    "dataset":
        "YelpChi",

    "mode":
        "UNIFIED_SMOKE",

    "ratio":
        "TR40",

    "split_seed":
        2,

    "training_seed":
        TRAIN_SEED,

    "epochs":
        1,

    "train_nodes":
        int(
            len(
                helper.train_nid
            )
        ),

    "validation_nodes":
        int(
            len(
                helper.val_nid
            )
        ),

    "test_nodes_registered":
        int(
            len(
                helper.test_nid
            )
        ),

    "gpu":
        GPU_NAME,

    "visible_gpu_count":
        int(
            torch.cuda.device_count()
        ),

    "optimizer":
        optimizer_name,

    "optimizer_betas":
        list(
            optimizer_betas
        ),

    "optimizer_eps":
        optimizer_eps,

    "learning_rate":
        optimizer_lr,

    "weight_decay":
        optimizer_weight_decay,

    "parameter_count":
        parameter_count,

    "trainable_parameter_count":
        trainable_parameter_count,

    "train_batches":
        num_train_batches,

    "total_train_loss":
        total_train_loss,

    "average_train_loss":
        average_train_loss,

    "train_seconds":
        float(
            train_seconds
        ),

    "peak_training_gpu_memory_mb":
        peak_train_memory_mb,

    "validation_inference_seconds":
        float(
            val_seconds
        ),

    "peak_validation_gpu_memory_mb":
        peak_val_memory_mb,

    "validation_auprc":
        common_val_auprc,

    "validation_auroc":
        common_val_auroc,

    "validation_macro_f1_at_0_5":
        common_val_macro_f1,

    "validation_fraud_precision_at_0_5":
        common_val_fraud_precision,

    "validation_fraud_recall_at_0_5":
        common_val_fraud_recall,

    "pmp_native_validation_auprc":
        pmp_val_auprc,

    "pmp_native_validation_auroc":
        pmp_val_auroc,

    "pmp_best_pr_threshold_diagnostic_only":
        float(
            pmp_val_results.best_pr_thres
        ),

    "smoke_threshold":
        0.5,

    "threshold_selected":
        False,

    "checkpoint_selected":
        False,

    "selection_metric_for_later_tuning":
        "validation AUPRC",

    "train_only_partition_labels_verified":
        True,

    "validation_evaluation_performed":
        True,

    "test_evaluation_performed":
        False,

    "architecture_modified":
        False
}


output_json.write_text(

    json.dumps(
        record,
        indent=2
    ),

    encoding="utf-8"
)


# ============================================================
# 23. PRINT RESULT
# ============================================================

print(
    "\n===== CHILD SMOKE RESULT =====",
    flush=True
)

print(
    "Validation AUPRC:",
    f"{common_val_auprc:.6f}",
    flush=True
)

print(
    "Validation AUROC:",
    f"{common_val_auroc:.6f}",
    flush=True
)

print(
    "Validation Macro-F1 @0.5:",
    f"{common_val_macro_f1:.6f}",
    flush=True
)

print(
    "Validation fraud precision @0.5:",
    f"{common_val_fraud_precision:.6f}",
    flush=True
)

print(
    "Validation fraud recall @0.5:",
    f"{common_val_fraud_recall:.6f}",
    flush=True
)

print(
    "Train seconds:",
    f"{train_seconds:.6f}",
    flush=True
)

print(
    "Validation inference seconds:",
    f"{val_seconds:.6f}",
    flush=True
)

print(
    "Peak training GPU memory:",
    f"{peak_train_memory_mb:.2f} MB",
    flush=True
)

print(
    "Peak validation GPU memory:",
    f"{peak_val_memory_mb:.2f} MB",
    flush=True
)

print(
    "Test evaluation performed: NO",
    flush=True
)
'''


SMOKE_SCRIPT.write_text(

    textwrap.dedent(
        smoke_code
    ).lstrip(),

    encoding="utf-8"
)


# ============================================================
# 3. RUN CHILD SMOKE
# ============================================================

cmd = [

    str(
        LAUNCHER
    ),

    str(
        SMOKE_SCRIPT
    ),

    str(
        REPO
    ),

    str(
        ADAPTER_DIR
    ),

    str(
        RAW_ROOT
    ),

    str(
        SPLIT_PATH
    ),

    str(
        SMOKE_JSON
    ),

    str(
        SMOKE_CONFIG
    )
]


print(
    "\n$",
    " ".join(
        cmd
    ),
    flush=True
)


p = subprocess.run(

    cmd,

    cwd=str(
        REPO
    ),

    text=True,

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    timeout=3600,

    check=False
)


SMOKE_LOG.write_text(
    p.stdout,
    encoding="utf-8"
)


print(
    p.stdout,
    flush=True
)


assert p.returncode == 0, (
    "Step 6B one-epoch smoke failed. "
    f"See {SMOKE_LOG}"
)


assert SMOKE_JSON.exists()


result = json.loads(
    SMOKE_JSON.read_text()
)


# ============================================================
# 4. RESULT SAFETY GATES
# ============================================================

assert result[
    "epochs"
] == 1


assert result[
    "training_seed"
] == 2


assert result[
    "visible_gpu_count"
] == 1


assert result[
    "train_only_partition_labels_verified"
] is True


assert result[
    "validation_evaluation_performed"
] is True


assert result[
    "test_evaluation_performed"
] is False


assert result[
    "checkpoint_selected"
] is False


assert result[
    "threshold_selected"
] is False


assert result[
    "architecture_modified"
] is False


# ============================================================
# 5. OFFICIAL TRACKED SOURCE MUST REMAIN UNCHANGED
# ============================================================

tracked_diff = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--quiet"
    ],
    check=False
)


staged_diff = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--cached",
        "--quiet"
    ],
    check=False
)


assert tracked_diff.returncode == 0, (
    "Tracked PMP source changed during smoke."
)


assert staged_diff.returncode == 0, (
    "Staged PMP source changed during smoke."
)


# ============================================================
# 6. REMOVE ONLY IMPORT-GENERATED CACHE
# ============================================================

for pth in sorted(
    REPO.rglob(
        "__pycache__"
    )
):

    if pth.is_dir():

        shutil.rmtree(
            pth
        )


for pattern in [
    "*.pyc",
    "*.pyo"
]:

    for pth in sorted(
        REPO.rglob(
            pattern
        )
    ):

        if pth.is_file():

            pth.unlink()


# ============================================================
# 7. NOTHING ELSE MAY HAVE APPEARED IN OFFICIAL REPO
# ============================================================

remaining_status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--porcelain=v1",
        "-uall"
    ],
    text=True
)


assert remaining_status.strip() == "", (
    "Unexpected repository residue after Step 6B:\n"
    + remaining_status
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n===== STEP 6B GATE =====",
    flush=True
)

print(
    "PASS — PMP × YelpChi TR40 one-epoch unified smoke completed.",
    flush=True
)

print(
    "Frozen project TR40 split used: YES",
    flush=True
)

print(
    "Training seed: 2",
    flush=True
)

print(
    "Training epochs: 1",
    flush=True
)

print(
    "PMP train-only partition labels preserved: YES",
    flush=True
)

print(
    "Adam beta1/beta2/eps verified: YES",
    flush=True
)

print(
    "Validation AUPRC:",
    f"{result['validation_auprc']:.6f}",
    flush=True
)

print(
    "Validation AUROC:",
    f"{result['validation_auroc']:.6f}",
    flush=True
)

print(
    "Validation Macro-F1 @0.5:",
    f"{result['validation_macro_f1_at_0_5']:.6f}",
    flush=True
)

print(
    "Training time:",
    f"{result['train_seconds']:.6f} s",
    flush=True
)

print(
    "Validation inference time:",
    f"{result['validation_inference_seconds']:.6f} s",
    flush=True
)

print(
    "Peak training GPU memory:",
    f"{result['peak_training_gpu_memory_mb']:.2f} MB",
    flush=True
)

print(
    "Test evaluation performed: NO",
    flush=True
)

print(
    "Checkpoint selected: NO",
    flush=True
)

print(
    "Threshold selected: NO",
    flush=True
)

print(
    "Official PMP source modified: NO",
    flush=True
)

print(
    "Architecture modified: NO",
    flush=True
)

print(
    "Ready for Step 7 TR40 tuning: YES",
    flush=True
)

## Step 7 — PMP × YelpChi TR40 Controlled Hyperparameter Tuning

Step 6B passed the one-epoch controlled smoke test.

The PMP training path is therefore eligible for the project's fixed TR40
tuning stage.

### Tuning protocol

- Dataset: YelpChi
- Model: PMP / LA-SAGE-S
- Split: frozen TR40
- Split seed: 2
- Training seed: 2 for every trial
- Maximum trials: 12
- Maximum epochs per trial: 100
- Early-stopping patience: 20 validation evaluations
- Validation frequency: every epoch
- Selection metric: validation AUPRC
- Test evaluation: forbidden
- GPU: controlled visible T4 GPU 0 only
- Optimizer: Adam with beta1=0.9, beta2=0.999, epsilon=1e-8

### Search-space policy

Only shared, implementation-exposed hyperparameters are varied:

- hidden dimension: 32, 64, 128
- dropout: 0.0, 0.3, 0.5
- learning rate: values within 1e-4 to 1e-2
- weight decay: 0, 1e-5, 1e-4, 1e-3

PMP-specific architecture settings remain frozen at the official Yelp
LA-SAGE-S values, including:

- one message-passing layer;
- full-neighbour operation;
- heterogeneous three-relation path;
- relation aggregation;
- residual coefficient;
- projection;
- transformation count.

This prevents tuning from redesigning PMP.

### Test isolation

The test node IDs remain registered for later final evaluation, but the test
DataLoader is replaced by a guard object during tuning. Any accidental attempt
to iterate it raises an error immediately.

### Evidence

Each trial saves:

- exact effective configuration;
- raw per-epoch training times;
- raw validation inference times;
- validation AUPRC/AUROC by epoch;
- best validation epoch;
- best checkpoint;
- peak GPU memory;
- complete trial summary.

The cell is resumable: a completely finished trial is not repeated if the
same cell must be rerun during the current Kaggle working session.

The test set is never evaluated in Step 7.

In [ ]:
# ============================================================
# STEP 7 — PMP × YELPCHI TR40 CONTROLLED TUNING
#
# 12 fixed trials
# TR40 only
# Training seed 2
# Max epochs 100
# Patience 20
# Selection: validation AUPRC
# Test access: FORBIDDEN
#
# This cell can take a while.
# It prints live progress and saves every completed trial.
# ============================================================

from pathlib import Path

import csv
import hashlib
import json
import shutil
import subprocess
import textwrap
import zipfile


print(
    "===== STEP 7 — PMP × YELPCHI TR40 TUNING =====",
    flush=True
)


# ============================================================
# 0. REQUIRE PREVIOUS STATE
# ============================================================

required_globals = [
    "ROOT",
    "DIRS",
    "REPO",
    "LAUNCHER",
    "ADAPTER_DIR",
    "RAW_ROOT",
    "SPLIT_PATH"
]


for name in required_globals:

    assert name in globals(), (
        f"{name} missing. "
        "Do not restart from Step 1 unless the Kaggle runtime "
        "itself has restarted."
    )


assert (
    DIRS["results_unified"]
    / "yelp"
    / "smoke"
    / "yelp_tr40_seed2_smoke.json"
).exists(), (
    "Step 6B smoke evidence is missing."
)


# ============================================================
# 1. DIRECTORIES
# ============================================================

TUNING_DIR = (
    DIRS["results_unified"]
    / "yelp"
    / "tuning"
)


TUNING_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRIALS_DIR = (
    TUNING_DIR
    / "trials"
)


TRIALS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TUNING_SCRIPT = (
    DIRS["evidence"]
    / "step7_yelp_tr40_tuning.py"
)


TUNING_LOG = (
    DIRS["logs"]
    / "step7_yelp_tr40_tuning.log"
)


FINAL_SUMMARY_JSON = (
    TUNING_DIR
    / "tuning_summary.json"
)


FINAL_SUMMARY_CSV = (
    TUNING_DIR
    / "tuning_summary.csv"
)


WINNER_JSON = (
    TUNING_DIR
    / "winner_config.json"
)


WINNER_YAML = (
    TUNING_DIR
    / "winner_config.yml"
)


# ============================================================
# 2. CHILD TUNING SCRIPT
# ============================================================

tuning_code = r'''
import copy
import csv
import gc
import hashlib
import json
import random
import sys
import time

from pathlib import Path
from types import SimpleNamespace

import dgl
import numpy as np
import torch
import yaml

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# ARGUMENTS
# ============================================================

repo = Path(
    sys.argv[1]
)

adapter_dir = Path(
    sys.argv[2]
)

raw_root = Path(
    sys.argv[3]
)

split_path = Path(
    sys.argv[4]
)

tuning_dir = Path(
    sys.argv[5]
)


trials_dir = (
    tuning_dir
    / "trials"
)


trials_dir.mkdir(
    parents=True,
    exist_ok=True
)


sys.path.insert(
    0,
    str(repo)
)

sys.path.insert(
    0,
    str(adapter_dir)
)


from DataHelper.datasetHelper import DatasetHelper
from training_procedure import Trainer

from pmp_unified_split_adapter import (
    apply_frozen_split
)


# ============================================================
# CONTROLLED PROTOCOL
# ============================================================

SCHEMA = (
    "COMP8851_PMP_YELP_TR40_TUNING_V1"
)

TRAIN_SEED = 2

MAX_EPOCHS = 100

PATIENCE = 20


# ============================================================
# 12 PREDECLARED TRIALS
#
# Architecture-specific PMP parameters remain frozen.
# Only shared exposed tuning dimensions change.
# ============================================================

TRIALS = [

    {
        "trial_id": 1,
        "hid_dim": 64,
        "dropout": 0.0,
        "lr": 1e-2,
        "weight_decay": 0.0
    },

    {
        "trial_id": 2,
        "hid_dim": 32,
        "dropout": 0.0,
        "lr": 1e-2,
        "weight_decay": 0.0
    },

    {
        "trial_id": 3,
        "hid_dim": 128,
        "dropout": 0.0,
        "lr": 1e-2,
        "weight_decay": 0.0
    },

    {
        "trial_id": 4,
        "hid_dim": 64,
        "dropout": 0.3,
        "lr": 1e-2,
        "weight_decay": 0.0
    },

    {
        "trial_id": 5,
        "hid_dim": 64,
        "dropout": 0.5,
        "lr": 1e-2,
        "weight_decay": 0.0
    },

    {
        "trial_id": 6,
        "hid_dim": 64,
        "dropout": 0.0,
        "lr": 5e-3,
        "weight_decay": 0.0
    },

    {
        "trial_id": 7,
        "hid_dim": 64,
        "dropout": 0.0,
        "lr": 1e-3,
        "weight_decay": 0.0
    },

    {
        "trial_id": 8,
        "hid_dim": 64,
        "dropout": 0.3,
        "lr": 5e-3,
        "weight_decay": 1e-5
    },

    {
        "trial_id": 9,
        "hid_dim": 32,
        "dropout": 0.3,
        "lr": 5e-3,
        "weight_decay": 1e-4
    },

    {
        "trial_id": 10,
        "hid_dim": 128,
        "dropout": 0.3,
        "lr": 5e-3,
        "weight_decay": 1e-4
    },

    {
        "trial_id": 11,
        "hid_dim": 64,
        "dropout": 0.5,
        "lr": 1e-3,
        "weight_decay": 1e-3
    },

    {
        "trial_id": 12,
        "hid_dim": 128,
        "dropout": 0.5,
        "lr": 1e-3,
        "weight_decay": 1e-5
    }
]


assert len(
    TRIALS
) == 12


# ============================================================
# DEVICE GATE
# ============================================================

assert torch.cuda.is_available()


assert torch.cuda.device_count() == 1, (
    "Controlled launcher must expose exactly one GPU."
)


torch.cuda.set_device(
    0
)


GPU_NAME = torch.cuda.get_device_name(
    0
)


assert "T4" in GPU_NAME.upper()


print(
    "Controlled GPU:",
    GPU_NAME,
    flush=True
)


# ============================================================
# SEED FUNCTION
# ============================================================

def set_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    torch.cuda.manual_seed_all(
        seed
    )

    dgl.seed(
        seed
    )

    try:

        dgl.random.seed(
            seed
        )

    except Exception:

        pass


    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# NULL LOGGER
# ============================================================

class NullLogger:

    def __init__(
        self
    ):

        self.append = ""


    def log(
        self,
        *args,
        **kwargs
    ):

        pass


    def add_line(
        self,
        *args,
        **kwargs
    ):

        pass


# ============================================================
# FORBIDDEN TEST LOADER
#
# Any accidental test iteration immediately fails Step 7.
# ============================================================

class ForbiddenTestLoader:

    def __iter__(
        self
    ):

        raise RuntimeError(
            "TEST ACCESS FORBIDDEN DURING TR40 TUNING"
        )


    def __len__(
        self
    ):

        raise RuntimeError(
            "TEST LOADER LENGTH ACCESS FORBIDDEN DURING TUNING"
        )


# ============================================================
# LOAD FROZEN OFFICIAL CONFIG
# ============================================================

CONFIG_PATH = (
    repo
    / "config"
    / "yelp.yml"
)


with CONFIG_PATH.open(
    "r"
) as f:

    config_root = yaml.safe_load(
        f
    )


assert "LA-SAGE-S" in config_root


official_cfg = dict(
    config_root[
        "LA-SAGE-S"
    ]
)


# ============================================================
# BASE UNIFIED CONFIG
# ============================================================

base_cfg = copy.deepcopy(
    official_cfg
)


base_cfg[
    "model_name"
] = "LA-SAGE-S"

base_cfg[
    "model"
] = "LA-SAGE-S"

base_cfg[
    "dataset"
] = "yelp"

base_cfg[
    "gpu_id"
] = 0

base_cfg[
    "seed"
] = TRAIN_SEED

base_cfg[
    "train_size"
] = 0.4

base_cfg[
    "val_size"
] = 0.2

base_cfg[
    "num_workers"
] = 0


# Unified checkpoint selection criterion.
base_cfg[
    "monitor"
] = "ap_gnn"


# Critical test isolation.
base_cfg[
    "test_each_epoch"
] = False


base_cfg[
    "epochs"
] = MAX_EPOCHS

base_cfg[
    "patience"
] = PATIENCE


# Evaluation predictions may use 0.5 internally,
# but tuning selection uses ranking AUPRC only.
base_cfg[
    "threshold_moving"
] = True

base_cfg[
    "thres"
] = 0.5


# ============================================================
# VERIFY ARCHITECTURE-SPECIFIC SETTINGS STAY FROZEN
# ============================================================

FROZEN_ARCH_KEYS = [

    "n_layer",
    "full_neighbors",
    "homo",
    "sampled_neighbors",
    "relation_agg",
    "resi",
    "proj",
    "num_trans",
    "agg",
    "reduction"
]


frozen_architecture = {

    key:
        official_cfg.get(
            key
        )

    for key
    in FROZEN_ARCH_KEYS
}


print(
    "\n===== FROZEN PMP ARCHITECTURE =====",
    flush=True
)


for key, value in frozen_architecture.items():

    print(
        f"{key}: {value}",
        flush=True
    )


# ============================================================
# LOAD YELPCHI ONCE
#
# This avoids repeating DGL dataset loading for every trial.
# ============================================================

set_seed(
    TRAIN_SEED
)


helper = DatasetHelper(
    base_cfg,
    dName="yelp"
)


helper.dataset_source_folder_path = str(
    raw_root
)


helper.load()


helper = apply_frozen_split(

    helper,

    split_path,

    ratio="TR40",

    num_workers=0
)


assert len(
    helper.train_nid
) == 18381

assert len(
    helper.val_nid
) == 9191

assert len(
    helper.test_nid
) == 18382


# ============================================================
# REVERIFY TRAIN-ONLY PMP PARTITION LABELS
# ============================================================

label_unk = (
    helper.data.ndata[
        "label_unk"
    ]
    .detach()
    .cpu()
)


labels_cpu = (
    helper.labels
    .detach()
    .cpu()
)


train_cpu = (
    helper.train_nid
    .detach()
    .cpu()
)


assert torch.equal(
    label_unk[
        train_cpu
    ],
    labels_cpu[
        train_cpu
    ]
)


outside_train = torch.ones(
    helper.num_nodes,
    dtype=torch.bool
)


outside_train[
    train_cpu
] = False


assert torch.all(
    label_unk[
        outside_train
    ]
    == 2
)


# Test loader becomes impossible to iterate.
helper.test_loader = ForbiddenTestLoader()


print(
    "\nFrozen TR40 split: VERIFIED",
    flush=True
)

print(
    "Train-only partition labels: VERIFIED",
    flush=True
)

print(
    "Test loader guard: ACTIVE",
    flush=True
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def to_numpy(
    value
):

    if torch.is_tensor(
        value
    ):

        return (
            value
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        value
    )


def canonical_hash(
    obj
):

    text = json.dumps(
        obj,
        sort_keys=True,
        separators=(
            ",",
            ":"
        )
    )

    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


# ============================================================
# PRINT PREDECLARED SEARCH
# ============================================================

print(
    "\n===== PREDECLARED 12 TRIALS =====",
    flush=True
)


for trial in TRIALS:

    print(
        "Trial "
        f"{trial['trial_id']:02d} | "
        f"hid={trial['hid_dim']} | "
        f"dropout={trial['dropout']} | "
        f"lr={trial['lr']} | "
        f"wd={trial['weight_decay']}",
        flush=True
    )


# ============================================================
# RUN / RESUME TRIALS
# ============================================================

trial_summaries = []


for trial in TRIALS:

    trial_id = int(
        trial[
            "trial_id"
        ]
    )


    trial_name = (
        f"trial_{trial_id:02d}"
    )


    trial_dir = (
        trials_dir
        / trial_name
    )


    trial_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    summary_path = (
        trial_dir
        / "summary.json"
    )


    epochs_path = (
        trial_dir
        / "epoch_times.csv"
    )


    config_path = (
        trial_dir
        / "config.yml"
    )


    checkpoint_path = (
        trial_dir
        / "best_validation_auprc.pth"
    )


    trial_signature = {

        "schema":
            SCHEMA,

        "dataset":
            "YelpChi",

        "ratio":
            "TR40",

        "split_seed":
            2,

        "training_seed":
            TRAIN_SEED,

        "max_epochs":
            MAX_EPOCHS,

        "patience":
            PATIENCE,

        "selection_metric":
            "validation AUPRC",

        "trial":
            trial
    }


    trial_signature_hash = (
        canonical_hash(
            trial_signature
        )
    )


    # --------------------------------------------------------
    # RESUME COMPLETED TRIAL
    # --------------------------------------------------------

    if (
        summary_path.exists()
        and
        epochs_path.exists()
        and
        checkpoint_path.exists()
    ):

        try:

            existing = json.loads(
                summary_path.read_text()
            )


            if (
                existing.get(
                    "status"
                )
                == "COMPLETE"

                and

                existing.get(
                    "trial_signature_sha256"
                )
                == trial_signature_hash
            ):

                print(
                    "\n"
                    f"===== TRIAL {trial_id:02d}/12 "
                    "— RESUME: ALREADY COMPLETE =====",
                    flush=True
                )


                print(
                    "Best epoch:",
                    existing[
                        "best_epoch"
                    ],
                    flush=True
                )

                print(
                    "Best validation AUPRC:",
                    f"{existing['best_val_auprc']:.6f}",
                    flush=True
                )


                trial_summaries.append(
                    existing
                )

                continue

        except Exception:

            pass


    # --------------------------------------------------------
    # EFFECTIVE TRIAL CONFIG
    # --------------------------------------------------------

    cfg = copy.deepcopy(
        base_cfg
    )


    cfg[
        "hid_dim"
    ] = int(
        trial[
            "hid_dim"
        ]
    )


    cfg[
        "dropout"
    ] = float(
        trial[
            "dropout"
        ]
    )


    cfg[
        "lr"
    ] = float(
        trial[
            "lr"
        ]
    )


    cfg[
        "weight_decay"
    ] = float(
        trial[
            "weight_decay"
        ]
    )


    cfg[
        "best_model_path"
    ] = str(
        checkpoint_path
    )


    # Architecture-specific values must not change.
    for key, expected in frozen_architecture.items():

        assert cfg.get(
            key
        ) == expected


    config_path.write_text(
        yaml.safe_dump(
            cfg,
            sort_keys=True
        ),
        encoding="utf-8"
    )


    print(
        "\n"
        "============================================================",
        flush=True
    )

    print(
        f"TRIAL {trial_id:02d}/12",
        flush=True
    )

    print(
        "============================================================",
        flush=True
    )

    print(
        f"hid_dim      = {cfg['hid_dim']}",
        flush=True
    )

    print(
        f"dropout      = {cfg['dropout']}",
        flush=True
    )

    print(
        f"lr           = {cfg['lr']}",
        flush=True
    )

    print(
        f"weight_decay = {cfg['weight_decay']}",
        flush=True
    )


    # --------------------------------------------------------
    # IDENTICAL TRAINING SEED FOR EVERY TRIAL
    # --------------------------------------------------------

    set_seed(
        TRAIN_SEED
    )


    # DatasetHelper config is aligned with the active trial.
    # Sampling/layer fields themselves remain frozen.
    helper.config = cfg


    logger = NullLogger()


    args = SimpleNamespace(

        gpu_id=0,

        seed=TRAIN_SEED,

        dataset="yelp",

        num_workers=0,

        train_size=0.4,

        val_size=0.2,

        multirun=1,

        run_best=False,

        data_dir=str(
            raw_root
        ),

        best_model_path=str(
            trial_dir
        )
    )


    T = Trainer(
        config=cfg,
        args=args,
        logger=logger
    )


    # --------------------------------------------------------
    # FRESH MODEL / OPTIMIZER FOR THIS TRIAL
    # --------------------------------------------------------

    model, optimizer, loss_func, scheduler = (
        T.init(
            helper
        )
    )


    assert (
        type(
            optimizer
        ).__name__.lower()
        == "adam"
    )


    opt_group = (
        optimizer.param_groups[
            0
        ]
    )


    assert tuple(
        opt_group[
            "betas"
        ]
    ) == (
        0.9,
        0.999
    )


    assert abs(
        float(
            opt_group[
                "eps"
            ]
        )
        - 1e-8
    ) < 1e-15


    assert abs(
        float(
            opt_group[
                "lr"
            ]
        )
        - cfg[
            "lr"
        ]
    ) < 1e-15


    assert abs(
        float(
            opt_group[
                "weight_decay"
            ]
        )
        - cfg[
            "weight_decay"
        ]
    ) < 1e-15


    parameter_count = int(
        sum(
            p.numel()
            for p in model.parameters()
        )
    )


    # --------------------------------------------------------
    # TRIAL STATE
    # --------------------------------------------------------

    best_val_auprc = float(
        "-inf"
    )

    best_val_auroc = float(
        "nan"
    )

    best_epoch = -1

    patience_count = 0

    total_training_seconds = 0.0

    total_validation_seconds = 0.0


    epoch_rows = []


    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats(
        0
    )


    # --------------------------------------------------------
    # EPOCH LOOP
    # --------------------------------------------------------

    for epoch in range(
        MAX_EPOCHS
    ):

        # ====================================================
        # TRAIN
        # ====================================================

        torch.cuda.synchronize(
            0
        )


        train_start = (
            time.perf_counter()
        )


        model, epoch_loss = T.train(

            epoch,

            model,

            loss_func,

            optimizer,

            helper.train_loader,

            helper
        )


        torch.cuda.synchronize(
            0
        )


        train_seconds = (
            time.perf_counter()
            - train_start
        )


        total_training_seconds += (
            train_seconds
        )


        if torch.is_tensor(
            epoch_loss
        ):

            total_loss = float(
                epoch_loss
                .detach()
                .cpu()
                .item()
            )

        else:

            total_loss = float(
                epoch_loss
            )


        average_loss = (
            total_loss
            / len(
                helper.train_loader
            )
        )


        # ====================================================
        # VALIDATION ONLY
        # ====================================================

        torch.cuda.synchronize(
            0
        )


        validation_start = (
            time.perf_counter()
        )


        val_labels, val_probs, _ = (
            T.evaluation(

                helper,

                helper.val_loader,

                model,

                threshold_moving=True,

                thres=0.5
            )
        )


        torch.cuda.synchronize(
            0
        )


        validation_seconds = (
            time.perf_counter()
            - validation_start
        )


        total_validation_seconds += (
            validation_seconds
        )


        y_val = to_numpy(
            val_labels
        ).reshape(
            -1
        )


        p_val = to_numpy(
            val_probs
        ).reshape(
            -1
        )


        assert len(
            y_val
        ) == 9191


        assert len(
            p_val
        ) == 9191


        assert np.isfinite(
            p_val
        ).all()


        val_auprc = float(
            average_precision_score(
                y_val,
                p_val
            )
        )


        val_auroc = float(
            roc_auc_score(
                y_val,
                p_val
            )
        )


        assert np.isfinite(
            val_auprc
        )

        assert np.isfinite(
            val_auroc
        )


        # ====================================================
        # VALIDATION-AUPRC CHECKPOINT SELECTION
        # ====================================================

        improved = (
            val_auprc
            > best_val_auprc
            + 1e-12
        )


        if improved:

            best_val_auprc = (
                val_auprc
            )

            best_val_auroc = (
                val_auroc
            )

            best_epoch = (
                epoch
            )

            patience_count = 0


            torch.save(

                {
                    "schema":
                        SCHEMA,

                    "trial_id":
                        trial_id,

                    "epoch":
                        epoch,

                    "validation_auprc":
                        val_auprc,

                    "validation_auroc":
                        val_auroc,

                    "training_seed":
                        TRAIN_SEED,

                    "model_state_dict":
                        model.state_dict()
                },

                checkpoint_path
            )

        else:

            patience_count += 1


        row = {

            "trial_id":
                trial_id,

            "epoch":
                epoch,

            "train_loss_total":
                total_loss,

            "train_loss_average":
                average_loss,

            "train_seconds":
                train_seconds,

            "validation_seconds":
                validation_seconds,

            "validation_auprc":
                val_auprc,

            "validation_auroc":
                val_auroc,

            "improved":
                int(
                    improved
                ),

            "patience_count":
                patience_count
        }


        epoch_rows.append(
            row
        )


        print(
            f"Trial {trial_id:02d} | "
            f"Epoch {epoch + 1:03d}/{MAX_EPOCHS} | "
            f"loss={average_loss:.5f} | "
            f"val_AP={val_auprc:.6f} | "
            f"val_AUC={val_auroc:.6f} | "
            f"train={train_seconds:.3f}s | "
            f"val={validation_seconds:.3f}s | "
            f"best={best_val_auprc:.6f} "
            f"@{best_epoch + 1:03d} | "
            f"pat={patience_count:02d}/{PATIENCE}",
            flush=True
        )


        # Save raw epoch evidence after every epoch so a log
        # is preserved even if later execution is interrupted.
        with epochs_path.open(
            "w",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=list(
                    epoch_rows[
                        0
                    ].keys()
                )
            )

            writer.writeheader()

            writer.writerows(
                epoch_rows
            )


        # ====================================================
        # EARLY STOP
        # ====================================================

        if patience_count >= PATIENCE:

            print(
                f"Trial {trial_id:02d}: "
                f"early stop after epoch {epoch + 1}; "
                f"best epoch={best_epoch + 1}, "
                f"best val AUPRC={best_val_auprc:.6f}",
                flush=True
            )

            break


    # --------------------------------------------------------
    # END TRIAL
    # --------------------------------------------------------

    assert best_epoch >= 0

    assert checkpoint_path.exists()


    peak_gpu_memory_mb = float(
        torch.cuda.max_memory_allocated(
            0
        )
        / (
            1024 ** 2
        )
    )


    completed_epochs = len(
        epoch_rows
    )


    average_epoch_train_seconds = float(
        np.mean(
            [
                x[
                    "train_seconds"
                ]
                for x
                in epoch_rows
            ]
        )
    )


    summary = {

        "schema":
            SCHEMA,

        "status":
            "COMPLETE",

        "trial_signature_sha256":
            trial_signature_hash,

        "trial_id":
            trial_id,

        "trial_config":
            trial,

        "dataset":
            "YelpChi",

        "ratio":
            "TR40",

        "split_seed":
            2,

        "training_seed":
            TRAIN_SEED,

        "selection_metric":
            "validation AUPRC",

        "max_epochs":
            MAX_EPOCHS,

        "patience":
            PATIENCE,

        "completed_epochs":
            completed_epochs,

        "best_epoch":
            best_epoch,

        "best_epoch_1_based":
            best_epoch + 1,

        "best_val_auprc":
            best_val_auprc,

        "best_val_auroc":
            best_val_auroc,

        "parameter_count":
            parameter_count,

        "total_training_seconds":
            total_training_seconds,

        "average_epoch_train_seconds":
            average_epoch_train_seconds,

        "total_validation_seconds":
            total_validation_seconds,

        "peak_gpu_memory_mb":
            peak_gpu_memory_mb,

        "best_checkpoint":
            str(
                checkpoint_path
            ),

        "epoch_times_csv":
            str(
                epochs_path
            ),

        "test_loader_guard_active":
            True,

        "test_evaluation_performed":
            False,

        "architecture_modified":
            False
    }


    summary_path.write_text(
        json.dumps(
            summary,
            indent=2
        ),
        encoding="utf-8"
    )


    trial_summaries.append(
        summary
    )


    print(
        "\n"
        f"TRIAL {trial_id:02d} COMPLETE | "
        f"epochs={completed_epochs} | "
        f"best epoch={best_epoch + 1} | "
        f"best val AUPRC={best_val_auprc:.6f} | "
        f"peak GPU={peak_gpu_memory_mb:.2f} MB",
        flush=True
    )


    # --------------------------------------------------------
    # GPU / OBJECT CLEANUP BEFORE NEXT TRIAL
    # --------------------------------------------------------

    del model
    del optimizer
    del loss_func
    del scheduler
    del T


    gc.collect()

    torch.cuda.empty_cache()


# ============================================================
# VERIFY ALL 12 TRIALS COMPLETED
# ============================================================

assert len(
    trial_summaries
) == 12


trial_summaries = sorted(
    trial_summaries,
    key=lambda x:
        x[
            "trial_id"
        ]
)


assert {
    x[
        "trial_id"
    ]
    for x
    in trial_summaries
} == set(
    range(
        1,
        13
    )
)


assert all(
    x[
        "test_evaluation_performed"
    ] is False
    for x in trial_summaries
)


# ============================================================
# WINNER
#
# Primary: highest validation AUPRC
# Exact tie: earliest predeclared trial ID
# ============================================================

winner = sorted(

    trial_summaries,

    key=lambda x: (
        -x[
            "best_val_auprc"
        ],
        x[
            "trial_id"
        ]
    )

)[
    0
]


winner_trial = winner[
    "trial_config"
]


winner_effective_cfg = copy.deepcopy(
    base_cfg
)


winner_effective_cfg[
    "hid_dim"
] = int(
    winner_trial[
        "hid_dim"
    ]
)

winner_effective_cfg[
    "dropout"
] = float(
    winner_trial[
        "dropout"
    ]
)

winner_effective_cfg[
    "lr"
] = float(
    winner_trial[
        "lr"
    ]
)

winner_effective_cfg[
    "weight_decay"
] = float(
    winner_trial[
        "weight_decay"
    ]
)


# Final runs will still control epochs/patience explicitly.
winner_effective_cfg[
    "epochs"
] = MAX_EPOCHS

winner_effective_cfg[
    "patience"
] = PATIENCE


# ============================================================
# SAVE GLOBAL SUMMARY
# ============================================================

summary_payload = {

    "schema":
        SCHEMA,

    "dataset":
        "YelpChi",

    "ratio":
        "TR40",

    "split_seed":
        2,

    "training_seed":
        TRAIN_SEED,

    "trial_count":
        12,

    "max_epochs":
        MAX_EPOCHS,

    "patience":
        PATIENCE,

    "selection_metric":
        "validation AUPRC",

    "test_evaluation_performed":
        False,

    "frozen_architecture":
        frozen_architecture,

    "trials":
        trial_summaries,

    "winner":
        winner,

    "winner_effective_config":
        winner_effective_cfg
}


(
    tuning_dir
    / "tuning_summary.json"
).write_text(

    json.dumps(
        summary_payload,
        indent=2
    ),

    encoding="utf-8"
)


# ============================================================
# SUMMARY CSV
# ============================================================

csv_rows = []


for item in trial_summaries:

    t = item[
        "trial_config"
    ]


    csv_rows.append(
        {

            "trial_id":
                item[
                    "trial_id"
                ],

            "hid_dim":
                t[
                    "hid_dim"
                ],

            "dropout":
                t[
                    "dropout"
                ],

            "lr":
                t[
                    "lr"
                ],

            "weight_decay":
                t[
                    "weight_decay"
                ],

            "completed_epochs":
                item[
                    "completed_epochs"
                ],

            "best_epoch":
                item[
                    "best_epoch_1_based"
                ],

            "best_val_auprc":
                item[
                    "best_val_auprc"
                ],

            "best_val_auroc":
                item[
                    "best_val_auroc"
                ],

            "avg_epoch_train_seconds":
                item[
                    "average_epoch_train_seconds"
                ],

            "peak_gpu_memory_mb":
                item[
                    "peak_gpu_memory_mb"
                ]
        }
    )


with (
    tuning_dir
    / "tuning_summary.csv"
).open(
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(
            csv_rows[
                0
            ].keys()
        )
    )

    writer.writeheader()

    writer.writerows(
        csv_rows
    )


# ============================================================
# FREEZE WINNER CONFIG
# ============================================================

winner_record = {

    "schema":
        SCHEMA,

    "dataset":
        "YelpChi",

    "frozen_after":
        "TR40 tuning",

    "selection_metric":
        "validation AUPRC",

    "winning_trial_id":
        winner[
            "trial_id"
        ],

    "best_validation_auprc":
        winner[
            "best_val_auprc"
        ],

    "best_validation_auroc":
        winner[
            "best_val_auroc"
        ],

    "best_epoch":
        winner[
            "best_epoch_1_based"
        ],

    "tuned_values":
        winner_trial,

    "effective_config":
        winner_effective_cfg,

    "reuse_without_retuning_for":
        [
            "TR40",
            "TR30",
            "TR20",
            "TR10"
        ],

    "final_training_seeds":
        [
            2,
            42,
            72
        ],

    "test_evaluation_during_tuning":
        False
}


(
    tuning_dir
    / "winner_config.json"
).write_text(

    json.dumps(
        winner_record,
        indent=2
    ),

    encoding="utf-8"
)


(
    tuning_dir
    / "winner_config.yml"
).write_text(

    yaml.safe_dump(
        winner_record,
        sort_keys=False
    ),

    encoding="utf-8"
)


# ============================================================
# FINAL CHILD OUTPUT
# ============================================================

print(
    "\n"
    "============================================================",
    flush=True
)

print(
    "STEP 7 CHILD TUNING COMPLETE",
    flush=True
)

print(
    "============================================================",
    flush=True
)

print(
    "Completed trials: 12/12",
    flush=True
)

print(
    "Test evaluations: 0",
    flush=True
)

print(
    "Winning trial:",
    winner[
        "trial_id"
    ],
    flush=True
)

print(
    "Winning hid_dim:",
    winner_trial[
        "hid_dim"
    ],
    flush=True
)

print(
    "Winning dropout:",
    winner_trial[
        "dropout"
    ],
    flush=True
)

print(
    "Winning lr:",
    winner_trial[
        "lr"
    ],
    flush=True
)

print(
    "Winning weight_decay:",
    winner_trial[
        "weight_decay"
    ],
    flush=True
)

print(
    "Winning best epoch:",
    winner[
        "best_epoch_1_based"
    ],
    flush=True
)

print(
    "Winning validation AUPRC:",
    f"{winner['best_val_auprc']:.6f}",
    flush=True
)

print(
    "Winning validation AUROC:",
    f"{winner['best_val_auroc']:.6f}",
    flush=True
)

print(
    "Winner frozen for lower ratios: YES",
    flush=True
)

print(
    "Test evaluation performed: NO",
    flush=True
)
'''


TUNING_SCRIPT.write_text(

    textwrap.dedent(
        tuning_code
    ).lstrip(),

    encoding="utf-8"
)


# ============================================================
# 3. RUN CHILD WITH LIVE OUTPUT
# ============================================================

cmd = [

    str(
        LAUNCHER
    ),

    str(
        TUNING_SCRIPT
    ),

    str(
        REPO
    ),

    str(
        ADAPTER_DIR
    ),

    str(
        RAW_ROOT
    ),

    str(
        SPLIT_PATH
    ),

    str(
        TUNING_DIR
    )
]


print(
    "\n$",
    " ".join(
        cmd
    ),
    flush=True
)


with TUNING_LOG.open(
    "w",
    encoding="utf-8"
) as log_file:

    p = subprocess.Popen(

        cmd,

        cwd=str(
            REPO
        ),

        text=True,

        stdout=subprocess.PIPE,

        stderr=subprocess.STDOUT,

        bufsize=1
    )


    assert p.stdout is not None


    for line in p.stdout:

        print(
            line,
            end="",
            flush=True
        )

        log_file.write(
            line
        )

        log_file.flush()


    returncode = p.wait()


assert returncode == 0, (
    "Step 7 tuning failed. "
    f"See {TUNING_LOG}"
)


# ============================================================
# 4. VERIFY OUTPUTS
# ============================================================

assert FINAL_SUMMARY_JSON.exists()

assert FINAL_SUMMARY_CSV.exists()

assert WINNER_JSON.exists()

assert WINNER_YAML.exists()


summary = json.loads(
    FINAL_SUMMARY_JSON.read_text()
)


winner = json.loads(
    WINNER_JSON.read_text()
)


assert summary[
    "trial_count"
] == 12


assert len(
    summary[
        "trials"
    ]
) == 12


assert summary[
    "test_evaluation_performed"
] is False


assert winner[
    "test_evaluation_during_tuning"
] is False


assert winner[
    "final_training_seeds"
] == [
    2,
    42,
    72
]


# ============================================================
# 5. OFFICIAL SOURCE MUST REMAIN UNCHANGED
# ============================================================

tracked_diff = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--quiet"
    ],
    check=False
)


staged_diff = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--cached",
        "--quiet"
    ],
    check=False
)


assert tracked_diff.returncode == 0

assert staged_diff.returncode == 0


# ============================================================
# 6. REMOVE ONLY GENERATED PYTHON CACHE
# ============================================================

for pth in sorted(
    REPO.rglob(
        "__pycache__"
    )
):

    if pth.is_dir():

        shutil.rmtree(
            pth
        )


for pattern in [
    "*.pyc",
    "*.pyo"
]:

    for pth in sorted(
        REPO.rglob(
            pattern
        )
    ):

        if pth.is_file():

            pth.unlink()


remaining_status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--porcelain=v1",
        "-uall"
    ],
    text=True
)


assert remaining_status.strip() == "", (
    "Unexpected official-repository residue after tuning:\n"
    + remaining_status
)


# ============================================================
# 7. PACKAGE TUNING EVIDENCE IMMEDIATELY
#
# This prevents later Kaggle cleanup from losing the tuning
# record.
# ============================================================

PACKAGE_DIR = (
    ROOT
    / "evidence_packages"
)


PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PACKAGE_PATH = (
    PACKAGE_DIR
    / "pmp_yelp_tr40_tuning_evidence_20260910.zip"
)


if PACKAGE_PATH.exists():

    PACKAGE_PATH.unlink()


with zipfile.ZipFile(
    PACKAGE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    # Tuning outputs.
    for path in TUNING_DIR.rglob(
        "*"
    ):

        if path.is_file():

            z.write(
                path,
                arcname=(
                    "tuning/"
                    + str(
                        path.relative_to(
                            TUNING_DIR
                        )
                    )
                )
            )


    # Key preceding evidence.
    evidence_files = [

        DIRS["evidence"]
        / "step5_yelpchi_dataset_gate.json",

        DIRS["evidence"]
        / "step6a_yelp_adapter_audit.json",

        DIRS["evidence"]
        / "step6a1_post_audit_recovery.json",

        DIRS["evidence"]
        / "step6a3_runtime_binding_audit.json",

        DIRS["results_unified"]
        / "yelp"
        / "smoke"
        / "yelp_tr40_seed2_smoke.json",

        TUNING_LOG
    ]


    for path in evidence_files:

        if path.exists():

            z.write(
                path,
                arcname=(
                    "supporting/"
                    + path.name
                )
            )


def sha256_file(
    path
):

    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:

        for block in iter(
            lambda:
                f.read(
                    8 * 1024 * 1024
                ),
            b""
        ):

            h.update(
                block
            )

    return h.hexdigest()


package_sha = sha256_file(
    PACKAGE_PATH
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n===== STEP 7 GATE =====",
    flush=True
)

print(
    "PASS — PMP × YelpChi TR40 controlled tuning completed.",
    flush=True
)

print(
    "Trials completed: 12/12",
    flush=True
)

print(
    "Training seed for tuning: 2",
    flush=True
)

print(
    "Maximum epochs per trial: 100",
    flush=True
)

print(
    "Early-stopping patience: 20",
    flush=True
)

print(
    "Checkpoint selection metric: validation AUPRC",
    flush=True
)

print(
    "Test evaluation performed: NO",
    flush=True
)

print(
    "Winning trial:",
    winner[
        "winning_trial_id"
    ],
    flush=True
)

print(
    "Winning hidden dimension:",
    winner[
        "tuned_values"
    ][
        "hid_dim"
    ],
    flush=True
)

print(
    "Winning dropout:",
    winner[
        "tuned_values"
    ][
        "dropout"
    ],
    flush=True
)

print(
    "Winning learning rate:",
    winner[
        "tuned_values"
    ][
        "lr"
    ],
    flush=True
)

print(
    "Winning weight decay:",
    winner[
        "tuned_values"
    ][
        "weight_decay"
    ],
    flush=True
)

print(
    "Winning validation AUPRC:",
    f"{winner['best_validation_auprc']:.6f}",
    flush=True
)

print(
    "Winning validation AUROC:",
    f"{winner['best_validation_auroc']:.6f}",
    flush=True
)

print(
    "Winning best epoch:",
    winner[
        "best_epoch"
    ],
    flush=True
)

print(
    "Winner frozen for TR40/TR30/TR20/TR10: YES",
    flush=True
)

print(
    "Final seeds reserved: 2, 42, 72",
    flush=True
)

print(
    "Official PMP source modified: NO",
    flush=True
)

print(
    "Architecture modified: NO",
    flush=True
)

print(
    "Tuning evidence package:",
    PACKAGE_PATH,
    flush=True
)

print(
    "Package SHA256:",
    package_sha,
    flush=True
)

print(
    "Ready for final three-seed controlled runs: YES",
    flush=True
)

## Step 7R — Kaggle Session Recovery Audit

The original Step 7 tuning process was interrupted when the Kaggle draft
session restarted.

This recovery step does not train anything.

It checks:

- whether the previous PMP workspace survived;
- whether the isolated PMP environment survived;
- whether the frozen repository and split survived;
- whether the original Step 7 child process is somehow still running;
- which tuning trials completed;
- whether Trial 2 contains partial epoch evidence;
- the last saved tuning-log lines.

No partial trial is automatically accepted as complete.

If an interrupted trial must be repeated, it will be restarted from epoch 1
with the same frozen configuration and seed so that the final trial remains
methodologically clean.

In [ ]:
# ============================================================
# STEP 7R — KAGGLE SESSION RECOVERY AUDIT
#
# NO TRAINING.
# ============================================================

from pathlib import Path

import csv
import json
import os
import subprocess


print(
    "===== STEP 7R — PMP TUNING RECOVERY AUDIT =====",
    flush=True
)


# ============================================================
# 1. RECONSTRUCT PATHS WITHOUT RELYING ON OLD NOTEBOOK GLOBALS
# ============================================================

ROOT = Path(
    "/kaggle/working/comp8851_pmp"
)

REPO = (
    ROOT
    / "source"
    / "PMP"
)

LAUNCHER = (
    ROOT
    / "pmp_python.sh"
)

SPLIT_PATH = (
    ROOT
    / "shared"
    / "splits"
    / "yelp_seed2_nested_splits.npz"
)

ADAPTER_DIR = (
    ROOT
    / "adapters"
)

RAW_ROOT = (
    ROOT
    / "shared"
    / "pmp_raw"
)

TUNING_DIR = (
    ROOT
    / "results"
    / "unified"
    / "yelp"
    / "tuning"
)

TRIALS_DIR = (
    TUNING_DIR
    / "trials"
)

TUNING_LOG = (
    ROOT
    / "logs"
    / "step7_yelp_tr40_tuning.log"
)


# ============================================================
# 2. WORKSPACE SURVIVAL
# ============================================================

checks = {

    "root":
        ROOT.exists(),

    "repo":
        REPO.exists(),

    "launcher":
        LAUNCHER.exists(),

    "split":
        SPLIT_PATH.exists(),

    "adapter":
        (
            ADAPTER_DIR
            / "pmp_unified_split_adapter.py"
        ).exists(),

    "raw_root":
        RAW_ROOT.exists(),

    "tuning_dir":
        TUNING_DIR.exists()
}


print(
    "\n===== WORKSPACE SURVIVAL =====",
    flush=True
)


for key, value in checks.items():

    print(
        f"{key:<12}: "
        + (
            "YES"
            if value
            else "NO"
        ),
        flush=True
    )


# ============================================================
# 3. CHECK WHETHER OLD TUNING PROCESS IS STILL ALIVE
# ============================================================

proc = subprocess.run(
    [
        "bash",
        "-lc",
        (
            "ps -eo pid,etimes,cmd | "
            "grep 'step7_yelp_tr40_tuning.py' | "
            "grep -v grep || true"
        )
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False
)


process_text = (
    proc.stdout.strip()
)


print(
    "\n===== EXISTING STEP 7 PROCESS =====",
    flush=True
)


if process_text:

    print(
        process_text,
        flush=True
    )

    old_process_alive = True

else:

    print(
        "NONE",
        flush=True
    )

    old_process_alive = False


# ============================================================
# 4. INSPECT SAVED TRIAL STATE
# ============================================================

print(
    "\n===== SAVED TRIAL STATE =====",
    flush=True
)


completed_trials = []

partial_trials = []


for trial_id in range(
    1,
    13
):

    trial_dir = (
        TRIALS_DIR
        / f"trial_{trial_id:02d}"
    )

    summary_path = (
        trial_dir
        / "summary.json"
    )

    epoch_path = (
        trial_dir
        / "epoch_times.csv"
    )

    checkpoint_path = (
        trial_dir
        / "best_validation_auprc.pth"
    )


    completed = False

    best_ap = None

    best_epoch = None

    saved_epochs = 0


    # --------------------------------------------------------
    # Completed trial
    # --------------------------------------------------------

    if summary_path.exists():

        try:

            summary = json.loads(
                summary_path.read_text()
            )


            if (
                summary.get(
                    "status"
                )
                == "COMPLETE"
            ):

                completed = True

                completed_trials.append(
                    trial_id
                )

                best_ap = summary.get(
                    "best_val_auprc"
                )

                best_epoch = summary.get(
                    "best_epoch_1_based"
                )

        except Exception:

            pass


    # --------------------------------------------------------
    # Epoch rows, including interrupted trial
    # --------------------------------------------------------

    if epoch_path.exists():

        try:

            with epoch_path.open(
                "r",
                newline="",
                encoding="utf-8"
            ) as f:

                rows = list(
                    csv.DictReader(
                        f
                    )
                )


            saved_epochs = len(
                rows
            )


            if rows:

                row_best = max(
                    rows,
                    key=lambda x:
                        float(
                            x[
                                "validation_auprc"
                            ]
                        )
                )


                if best_ap is None:

                    best_ap = float(
                        row_best[
                            "validation_auprc"
                        ]
                    )


                if best_epoch is None:

                    best_epoch = (
                        int(
                            row_best[
                                "epoch"
                            ]
                        )
                        + 1
                    )

        except Exception as e:

            print(
                f"Trial {trial_id:02d} "
                f"CSV read warning: {e}",
                flush=True
            )


    if (
        not completed
        and
        saved_epochs > 0
    ):

        partial_trials.append(
            trial_id
        )


    state = (
        "COMPLETE"
        if completed
        else (
            "PARTIAL"
            if saved_epochs > 0
            else "NOT STARTED"
        )
    )


    print(
        f"Trial {trial_id:02d} | "
        f"{state:<11} | "
        f"saved_epochs={saved_epochs:>3} | "
        f"best_epoch={str(best_epoch):>4} | "
        f"best_AP={best_ap}",
        flush=True
    )


# ============================================================
# 5. SHOW LAST LOG LINES
# ============================================================

print(
    "\n===== LAST STEP 7 LOG LINES =====",
    flush=True
)


if TUNING_LOG.exists():

    lines = TUNING_LOG.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()


    for line in lines[
        -30:
    ]:

        print(
            line,
            flush=True
        )

else:

    print(
        "Tuning log does not exist.",
        flush=True
    )


# ============================================================
# 6. REPOSITORY COMMIT IF AVAILABLE
# ============================================================

repo_commit = None


if (
    REPO
    / ".git"
).exists():

    p = subprocess.run(
        [
            "git",
            "-C",
            str(REPO),
            "rev-parse",
            "HEAD"
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False
    )


    if p.returncode == 0:

        repo_commit = (
            p.stdout.strip()
        )


print(
    "\nRepository commit:",
    repo_commit,
    flush=True
)


# ============================================================
# 7. RECOVERY DECISION
# ============================================================

critical_workspace_ok = all(
    [
        checks[
            "root"
        ],
        checks[
            "repo"
        ],
        checks[
            "launcher"
        ],
        checks[
            "split"
        ],
        checks[
            "adapter"
        ]
    ]
)


print(
    "\n===== STEP 7R GATE =====",
    flush=True
)


if old_process_alive:

    print(
        "HOLD — previous Step 7 process is still running.",
        flush=True
    )

    print(
        "Do NOT start another tuning process.",
        flush=True
    )

    print(
        "Use the saved log to monitor it.",
        flush=True
    )


elif critical_workspace_ok:

    print(
        "PASS — PMP workspace survived the session interruption.",
        flush=True
    )

    print(
        "Old Step 7 process running: NO",
        flush=True
    )

    print(
        "Completed trials:",
        completed_trials,
        flush=True
    )

    print(
        "Interrupted/partial trials:",
        partial_trials,
        flush=True
    )

    print(
        "Completed trials will be preserved: YES",
        flush=True
    )

    print(
        "Partial trial will be accepted as complete: NO",
        flush=True
    )

    print(
        "Safe recovery possible without rerunning Steps 1–6: YES",
        flush=True
    )


else:

    print(
        "FAIL — critical /kaggle/working PMP state was lost.",
        flush=True
    )

    print(
        "Do NOT run Step 7.",
        flush=True
    )

    print(
        "A controlled environment/state rebuild is required.",
        flush=True
    )